# <font color="red">**Higher-Dimensional Optical Data Visualization Challenge**</font>

## Composition → spectrum → measurement context

### The problem

Optical libraries are difficult to visualize because every sample has a composition,
a full spectrum, a measurement geometry, and sometimes an elapsed-time history. A
collection of separate plots makes comparisons slow, but forcing everything into one
crowded plot can hide the spectral evidence.

> **Can we make all of those variables easier to explore without hiding the spectra
> or compressing the experiment into a single unexplained score?**

### The approach

Like the example EELS hackathon notebook, this notebook follows one linear pattern:

1. inspect the data and its dimensions;
2. run one complete, deliberately simple worked example;
3. modify that baseline in a clearly marked challenge cell;
4. test whether a new user can answer the stated scientific question.

### Choose one challenge

1. **Challenge 1 — Quaternary System:** connect FA/Cs/Rb/DMA composition and I/Br variation to
   Top PL, Bottom PL, and absorbance.
2. **Challenge 2 — RP-Additive System:** connect 3AMP/FAPbI3 host composition and nominal MACl
   level to Top/Bottom PL over 0–380 minutes.

### Your final result

- one visualization that answers a specific scientific question;
- the full spectral evidence remains reachable or visible;
- one sentence explaining what became easier to see and one possible misinterpretation.

The worked examples are deliberately simple baselines not
the expected final answer.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kbarakati/athena_camm_hackathon/blob/k4my4r/docs/day_19_02102026/Higher_Dimensional_Optical_Data_Hackathon.ipynb)

<h2><font color="purple">1.</font> Download and Load the Data</h2>

Run the next cells from top to bottom. One public Google Drive folder contains both
challenge folders. The download cell is safe to rerun: it reuses a verified local copy
and only downloads again when files are missing or changed.


In [1]:
# @title 1. Install and import the small analysis stack
%pip -q install "gdown==6.4.1" "pyarrow>=14" "plotly>=5.22"

from html import escape
from pathlib import Path
import hashlib
import shutil
import sys

import gdown
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import FileLink, HTML, Markdown, display


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 1.9 MB/s eta 0:00:00


In [2]:
# @title 2. Download both challenge folders from Google Drive
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/16YfPjFdZHgW3hkC0QBLrCS9um2C6aK5q?usp=drive_link"
IN_COLAB = "google.colab" in sys.modules
DATA_ROOT = Path(
    "/content/higher_dimensional_optical_data"
    if IN_COLAB else "./.higher_dimensional_optical_data"
).resolve()
STAGING_ROOT = DATA_ROOT.with_name(DATA_ROOT.name + "_download")

REMOTE_QUATERNARY_FOLDER = "01_hochan_quaternary_data"
EXPECTED_MANIFEST_HASHES = {
    REMOTE_QUATERNARY_FOLDER: "6c4f5329411f7897b16578ec3f029f092ca05eb99a884a9a622b1ffabf24ff8e",
    "02_rp_additive_data": "0060b641f417179122562eefb6d5acee527b4927796fd4a595631b683fded38c",
}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def folder_errors(root):
    errors = []
    for folder_name, expected_manifest_hash in EXPECTED_MANIFEST_HASHES.items():
        folder = root / folder_name
        checksum_path = folder / "SHA256SUMS.txt"
        if not checksum_path.is_file():
            errors.append(f"missing: {folder_name}/SHA256SUMS.txt")
            continue
        if file_sha256(checksum_path) != expected_manifest_hash:
            errors.append(f"unexpected checksum manifest: {folder_name}")
            continue
        for line in checksum_path.read_text(encoding="utf-8").strip().splitlines():
            expected, relative = line.split("  ", 1)
            path = folder / relative
            if not path.is_file():
                errors.append(f"missing: {folder_name}/{relative}")
            elif file_sha256(path) != expected:
                errors.append(f"checksum mismatch: {folder_name}/{relative}")
    return errors

def locate_download_root(root):
    candidates = [root] + sorted(
        (path for path in root.rglob("*") if path.is_dir()),
        key=lambda path: len(path.parts),
    )
    matches = [
        path for path in candidates
        if (path / REMOTE_QUATERNARY_FOLDER).is_dir()
        and (path / "02_rp_additive_data").is_dir()
    ]
    if len(matches) != 1:
        raise RuntimeError(
            "Expected one downloaded parent containing both challenge folders; "
            f"found {len(matches)}."
        )
    return matches[0]

errors = folder_errors(DATA_ROOT)
if errors:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT)
    STAGING_ROOT.mkdir(parents=True)
    print("Downloading both challenge datasets from Google Drive ...")
    try:
        downloaded = gdown.download_folder(
            url=DRIVE_FOLDER_URL,
            output=str(STAGING_ROOT),
            quiet=False,
            use_cookies=False,
        )
        if not downloaded:
            raise RuntimeError("Google Drive returned no files.")
        downloaded_root = locate_download_root(STAGING_ROOT)
        errors = folder_errors(downloaded_root)
        if errors:
            raise RuntimeError("Downloaded data failed verification:\n- " + "\n- ".join(errors))

        if DATA_ROOT.exists():
            shutil.rmtree(DATA_ROOT)
        if downloaded_root == STAGING_ROOT:
            STAGING_ROOT.rename(DATA_ROOT)
        else:
            shutil.move(str(downloaded_root), str(DATA_ROOT))
            shutil.rmtree(STAGING_ROOT, ignore_errors=True)
    except Exception as exc:
        shutil.rmtree(STAGING_ROOT, ignore_errors=True)
        raise RuntimeError(
            "Data setup failed. Confirm that the shared parent folder and its contents "
            "are set to 'Anyone with the link - Viewer', then rerun this cell."
        ) from exc
    print("Download complete; every file passed SHA-256 verification.")
else:
    print("Verified data already present; no download needed.")

QUATERNARY_DIR = DATA_ROOT / REMOTE_QUATERNARY_FOLDER
RP_DIR = DATA_ROOT / "02_rp_additive_data"
print("Challenge 1 — Quaternary System: ready")
print("Challenge 2 — RP-Additive System: ready")


Retrieving folder contents


Retrieving folder 1lKab_j038FiNxPvNzHtnMUv_U9es1gN7 01_hochan_quaternary_data
Processing file 1Z20at8pdv7XnxWKYNDkNUDtzwHMHnrOu composition_and_recipe.csv
Processing file 1lIYfpqZTv7B1rU9QVv3zWyK7QP_xaZn6 measurement_manifest.csv
Processing file 16JOB1gy7TrfBvBp68hgJR4FcQjw_c5lZ optical_read_qc_summary.csv
Processing file 1VIJhV0YhdqfpmMIZxljMzWyOwWc1pYnz optical_spectra_long.parquet
Processing file 19Gmib_Jf0RhhJCsw_R9oxpNxfQmQG-9g README_DATA.md
Processing file 1h-cmt1usy-jDRkRNU3VqWZtR_Xo7DFOW SHA256SUMS.txt
Retrieving folder 1XCDHKsa7-_pmEmpH1EPpArxUlf54HvIH 02_rp_additive_data
Processing file 1bTvZUNQbc7FtEtNSXDBBGGoe000OPvcz composition_and_recipe.csv
Processing file 1qJ8wxyUdn_1rSESAV8bm_LJcsIBK0QJV metadata_features.parquet
Processing file 1Rz8SY732Cbf7oONnmumZcxE8Dt2a1lmt README_DATA.md
Processing file 1Rl82CZGXM248CoWYgy9KmJvBzspEg5rm SHA256SUMS.txt
Processing file 1nGx6ng2If1IlFK_l4lSGXlLzbgbWR1Lc spectra_wide.parquet


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1Z20at8pdv7XnxWKYNDkNUDtzwHMHnrOu
To: /content/higher_dimensional_optical_data_download/01_hochan_quaternary_data/composition_and_recipe.csv
100%|██████████| 20.5k/20.5k [00:00<00:00, 35.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1lIYfpqZTv7B1rU9QVv3zWyK7QP_xaZn6
To: /content/higher_dimensional_optical_data_download/01_hochan_quaternary_data/measurement_manifest.csv
100%|██████████| 692/692 [00:00<00:00, 2.47MB/s]
Downloading...
From: https://drive.google.com/uc?id=16JOB1gy7TrfBvBp68hgJR4FcQjw_c5lZ
To: /content/higher_dimensional_optical_data_download/01_hochan_quaternary_data/optical_read_qc_summary.csv
100%|██████████| 630/630 [00:00<00:00, 2.22MB/s]
Downloading...
From: https://drive.google.com/uc?id=1VIJhV0YhdqfpmMIZxljMzWyOwWc1pYnz
To: /content/higher_dimensional_optical_data_download/01_hochan_quaternary_data/

Download complete; every file passed SHA-256 verification.
Challenge 1 — Quaternary System: ready
Challenge 2 — RP-Additive System: ready



Download completed


In [3]:
# @title 3. Load the prepared tables
QUATERNARY_COMPOSITION_PATH = QUATERNARY_DIR / "composition_and_recipe.csv"
RP_COMPOSITION_PATH = RP_DIR / "composition_and_recipe.csv"

quaternary_composition = pd.read_csv(QUATERNARY_COMPOSITION_PATH)
quaternary_optical = pd.read_parquet(QUATERNARY_DIR / "optical_spectra_long.parquet")
quaternary_manifest = pd.read_csv(QUATERNARY_DIR / "measurement_manifest.csv")

rp_composition = pd.read_csv(RP_COMPOSITION_PATH)
rp_features = pd.read_parquet(RP_DIR / "metadata_features.parquet")
rp_spectra = pd.read_parquet(RP_DIR / "spectra_wide.parquet")

rp_wavelength_columns = sorted(
    (column for column in rp_spectra if column.startswith("wl_")),
    key=lambda column: float(column.split("_", 1)[1]),
)
rp_wavelengths = np.asarray([
    float(column.split("_", 1)[1]) for column in rp_wavelength_columns
])
rp_spectra_by_id = rp_spectra.set_index("spectrum_id")


In [4]:
# @title 4. Run compact scientific-structure checks
assert len(quaternary_composition) == quaternary_composition["well"].nunique() == 96
assert set(quaternary_composition["well"]) == set(quaternary_optical["well"])
assert set(quaternary_optical["geometry"]) == {"Top", "Bottom", "Absorbance"}

a_site = ["FA_fraction", "Cs_fraction", "Rb_fraction", "DMA_fraction"]
x_site = ["I_fraction", "Br_fraction", "Cl_fraction"]
assert np.allclose(quaternary_composition[a_site].sum(axis=1), 1.0)
assert np.allclose(quaternary_composition[x_site].sum(axis=1), 1.0)

assert len(rp_composition) == rp_composition["well"].nunique() == 96
assert set(rp_composition["well"]) == set(rp_features["well"])
assert len(rp_features) == rp_features["spectrum_id"].nunique() == 3840
assert set(rp_features["spectrum_id"]) == set(rp_spectra["spectrum_id"])
assert np.allclose(
    rp_composition["spacer_host_fraction"]
    + rp_composition["FAPbI3_host_fraction"],
    1.0,
)

coverage = pd.DataFrame([
    {
        "challenge": "Challenge 1 — Quaternary System",
        "wells": quaternary_composition["well"].nunique(),
        "spectra": quaternary_optical.groupby(["well", "geometry"]).ngroups,
        "times": "none",
        "measurements": "Top PL, Bottom PL, absorbance",
    },
    {
        "challenge": "Challenge 2 — RP-Additive System",
        "wells": rp_composition["well"].nunique(),
        "spectra": len(rp_features),
        "times": rp_features["time_min"].nunique(),
        "measurements": "Top PL, Bottom PL",
    },
])
display(Markdown("### Data ready — all integrity and structure checks passed"))
display(coverage)


### Data ready — all integrity and structure checks passed

,challenge,wells,spectra,times,measurements
0,Challenge 1 — Quaternary System,96,288,none,"Top PL, Bottom PL, absorbance"
1,Challenge 2 — RP-Additive System,96,3840,20,"Top PL, Bottom PL"


## What is “high-dimensional” here?

| Challenge | Composition variables | Measurement variables | Spectral variable |
|---|---|---|---|
| Quaternary System | FA, Cs, Rb, DMA; I/Br slice | Top PL, Bottom PL, absorbance | wavelength |
| RP-Additive System | 3AMP/FAPbI3 host fraction; nominal MACl level | time; Top/Bottom PL | wavelength |

Two guardrails matter throughout:

- In the Quaternary System, **Top and Bottom are collection geometries**, not time
  points or replicates.
- In the RP-Additive System, MACl percentage is a **nominal design level with
  undocumented basis**,
  not a third closed-composition fraction.

PL peaks and spectral summaries are optical observations—not structural phase labels.

### Possible visualization strategies

There is no single required chart type. Useful solution families include:

- **small multiples:** reserve panels for composition slices, geometry, or time;
- **overview + detail:** use a composition map to navigate to complete spectra;
- **interactive controls:** let the user select a well, time, geometry, or metric;
- **change encodings:** show Top − Bottom or final − initial values with a shared scale;
- **spectral-first views:** use heatmaps, trajectories, or compact spectral glyphs.

The worked examples below use **overview + detail**. They are starting points, not the
intended final answer. Figures are stacked at full width by default so labels, legends,
and color scales stay readable in Colab.


In [5]:
# Inspect or download the two one-row-per-well composition tables
display(Markdown("#### Challenge 1 — Quaternary System composition and recipe"))
display(quaternary_composition.head(6))
display(FileLink(
    str(QUATERNARY_COMPOSITION_PATH), result_html_prefix="Download full CSV: "
))

display(Markdown("#### Challenge 2 — RP-Additive System composition and recipe"))
display(rp_composition.head(6))
display(FileLink(str(RP_COMPOSITION_PATH), result_html_prefix="Download full CSV: "))


#### Challenge 1 — Quaternary System composition and recipe

,well,row,col,FA_fraction,Cs_fraction,Rb_fraction,DMA_fraction,I_fraction,Br_fraction,Cl_fraction,...,Br_design_percent,total_volume_uL,FAPbI3_volume_uL,FAPbBr3_volume_uL,RbPbI3_volume_uL,DMAPbI3_volume_uL,CsPbI3_volume_uL,CsPbI05Br05_volume_uL,composition_label,composition_basis
0,A1,A,1,0.900,0.1,0.0,0.000,1.0000,0.0000,0.0,...,0.000000,100.0,90.0,0.0,0.0,0.0,10.0,0.0,FA 0.900 | Cs 0.100 | Rb 0.000 | DMA 0.000 || ...,nominal_equal_molar_stock_volume
1,A2,A,2,0.875,0.1,0.0,0.025,1.0000,0.0000,0.0,...,0.000000,100.0,87.5,0.0,0.0,2.5,10.0,0.0,FA 0.875 | Cs 0.100 | Rb 0.000 | DMA 0.025 || ...,nominal_equal_molar_stock_volume
2,A3,A,3,0.850,0.1,0.0,0.050,1.0000,0.0000,0.0,...,0.000000,100.0,85.0,0.0,0.0,5.0,10.0,0.0,FA 0.850 | Cs 0.100 | Rb 0.000 | DMA 0.050 || ...,nominal_equal_molar_stock_volume
3,A4,A,4,0.800,0.1,0.0,0.100,1.0000,0.0000,0.0,...,0.000000,100.0,80.0,0.0,0.0,10.0,10.0,0.0,FA 0.800 | Cs 0.100 | Rb 0.000 | DMA 0.100 || ...,nominal_equal_molar_stock_volume
4,A5,A,5,0.900,0.1,0.0,0.000,0.8335,0.1665,0.0,...,16.666667,100.0,75.0,15.0,0.0,0.0,6.7,3.3,FA 0.900 | Cs 0.100 | Rb 0.000 | DMA 0.000 || ...,nominal_equal_molar_stock_volume
5,A6,A,6,0.875,0.1,0.0,0.025,0.8335,0.1665,0.0,...,16.666667,100.0,72.5,15.0,0.0,2.5,6.7,3.3,FA 0.875 | Cs 0.100 | Rb 0.000 | DMA 0.025 || ...,nominal_equal_molar_stock_volume


/content/higher_dimensional_optical_data/01_hochan_quaternary_data/composition_and_recipe.csv

#### Challenge 2 — RP-Additive System composition and recipe

,well,row,col,library,base_spacer,additive,spacer_host_fraction,FAPbI3_host_fraction,spacer_percent,FAPbI3_percent,additive_percent,spacer_stock_volume_uL,FAPbI3_stock_volume_uL,additive_stock_volume_uL,host_total_uL,total_mix_volume_uL,composition_label,composition_basis
0,A1,A,1,3AMP-FA with MACl,3AMP,MACl,1.000000,0.000000,100.000000,0.000000,0.0,50.00,0.00,0.0,50.0,50.0,3AMP:1.00 FA:0.00 MACl:0%,host_fraction_from_nominal_stock_volume;additi...
1,A2,A,2,3AMP-FA with MACl,3AMP,MACl,0.909091,0.090909,90.909091,9.090909,0.0,45.45,4.55,0.0,50.0,50.0,3AMP:0.91 FA:0.09 MACl:0%,host_fraction_from_nominal_stock_volume;additi...
2,A3,A,3,3AMP-FA with MACl,3AMP,MACl,0.818182,0.181818,81.818182,18.181818,0.0,40.91,9.09,0.0,50.0,50.0,3AMP:0.82 FA:0.18 MACl:0%,host_fraction_from_nominal_stock_volume;additi...
3,A4,A,4,3AMP-FA with MACl,3AMP,MACl,0.727273,0.272727,72.727273,27.272727,0.0,36.36,13.64,0.0,50.0,50.0,3AMP:0.73 FA:0.27 MACl:0%,host_fraction_from_nominal_stock_volume;additi...
4,A5,A,5,3AMP-FA with MACl,3AMP,MACl,0.636364,0.363636,63.636364,36.363636,0.0,31.82,18.18,0.0,50.0,50.0,3AMP:0.64 FA:0.36 MACl:0%,host_fraction_from_nominal_stock_volume;additi...
5,A6,A,6,3AMP-FA with MACl,3AMP,MACl,0.545455,0.454545,54.545455,45.454545,0.0,27.27,22.73,0.0,50.0,50.0,3AMP:0.55 FA:0.45 MACl:0%,host_fraction_from_nominal_stock_volume;additi...


/content/higher_dimensional_optical_data/02_rp_additive_data/composition_and_recipe.csv

In [6]:
# Responsive display/export helpers used by both worked examples.
# One column is the readable Colab default; use columns=2 only on a wide display.
def show_figures(figures, columns=1):
    if columns not in (1, 2):
        raise ValueError("columns must be 1 or 2")
    blocks = [
        "<div style='min-width:0;width:100%;overflow-x:auto'>"
        + pio.to_html(
            figure,
            include_plotlyjs="cdn" if index == 0 else False,
            full_html=False,
            config={"responsive": True, "displaylogo": False},
        )
        + "</div>"
        for index, figure in enumerate(figures)
    ]
    grid_class = f"hd-figure-grid-{columns}"
    display(HTML(
        f"<style>.{grid_class}{{display:grid;grid-template-columns:repeat("
        f"{columns},minmax(0,1fr));gap:28px;align-items:start;width:100%;"
        "max-width:1200px;margin:0 auto}}"
        f"@media(max-width:1100px){{.{grid_class}{{grid-template-columns:1fr}}}}"
        "</style>"
        f"<div class='{grid_class}'>" + "".join(blocks) + "</div>"
    ))

def export_figure_grid(figures, path, title, columns=1):
    if columns not in (1, 2):
        raise ValueError("columns must be 1 or 2")
    blocks = [
        pio.to_html(
            figure,
            include_plotlyjs=True if index == 0 else False,
            full_html=False,
            config={"responsive": True, "displaylogo": False},
        )
        for index, figure in enumerate(figures)
    ]
    document = (
        "<!doctype html><meta charset='utf-8'><title>" + escape(title) + "</title>"
        "<style>body{font-family:Arial;margin:24px;color:#172b4d}"
        ".grid{display:grid;gap:28px;max-width:1200px;margin:0 auto}"
        f".grid{{grid-template-columns:repeat({columns},minmax(0,1fr))}}"
        "@media(max-width:1100px){.grid{grid-template-columns:1fr}}"
        ".grid>div{min-width:0;width:100%;overflow-x:auto}</style>"
        "<h1>" + escape(title) + "</h1><div class='grid'>"
        + "".join("<div>" + block + "</div>" for block in blocks)
        + "</div>"
    )
    Path(path).write_text(document, encoding="utf-8")
    return Path(path)


<h2><font color="purple">2.</font> Explore the Quaternary System</h2>

### Problem 1

> **Where do Top and Bottom PL disagree across composition, and what additional
> context does absorbance provide?**

The A-site fractions **FA + Cs + Rb + DMA = 1**. The X-site fractions
**I + Br + Cl = 1** separately. Never normalize all seven fractions together.

A conventional ternary diagram is not enough because the A-site contains four
components and the plate also varies halide composition. The baseline solution uses
**small-multiple composition slices** for overview and a separate **full-spectrum
detail view** for evidence.

We will first create a small well-level feature table. These scalar features are
navigation aids; the full spectra remain the evidence.


In [7]:
# Summarize each Quaternary System PL spectrum without changing the source data
integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

def summarize_pl(group):
    ordered = group.sort_values("wavelength_nm")
    wavelength = ordered["wavelength_nm"].to_numpy(float)
    raw = ordered["optical_signal_instrument_units"].to_numpy(float)
    signal = np.clip(raw - np.nanpercentile(raw, 5), 0, None)
    area = float(integrate(signal, wavelength))
    peak = float(wavelength[int(np.nanargmax(signal))])
    centroid = float(np.sum(wavelength * signal) / signal.sum())
    return pd.Series({
        "dominant_peak_nm": peak,
        "centroid_nm": centroid,
        "log10_integrated_pl": np.log10(area),
    })

quaternary_pl_features = (
    quaternary_optical[quaternary_optical["modality"].eq("PL")]
    .groupby(["well", "geometry"], sort=False)[
        ["wavelength_nm", "optical_signal_instrument_units"]
    ]
    .apply(summarize_pl)
    .reset_index()
)


In [8]:
# Assemble one row per well for composition-space visualization
pl_wide = quaternary_pl_features.pivot(
    index="well", columns="geometry",
    values=["dominant_peak_nm", "centroid_nm", "log10_integrated_pl"],
)
pl_wide.columns = [
    f"{geometry}_{metric}" for metric, geometry in pl_wide.columns
]
pl_wide = pl_wide.reset_index()

absorbance_points = (
    quaternary_optical[
        quaternary_optical["modality"].eq("Absorbance")
        & quaternary_optical["wavelength_nm"].isin([450, 650, 750])
    ]
    .pivot(index="well", columns="wavelength_nm",
           values="optical_signal_instrument_units")
    .rename(columns=lambda value: f"Abs_signal_{int(value)}nm")
    .reset_index()
)

quaternary_features = (
    quaternary_composition.merge(pl_wide, on="well", validate="one_to_one")
    .merge(absorbance_points, on="well", validate="one_to_one")
)
quaternary_features["Top_minus_Bottom_peak_nm"] = (
    quaternary_features["Top_dominant_peak_nm"]
    - quaternary_features["Bottom_dominant_peak_nm"]
)
quaternary_features["Top_minus_Bottom_centroid_nm"] = (
    quaternary_features["Top_centroid_nm"]
    - quaternary_features["Bottom_centroid_nm"]
)
display(quaternary_features.head())


,well,row,col,FA_fraction,Cs_fraction,Rb_fraction,DMA_fraction,I_fraction,Br_fraction,Cl_fraction,...,Top_dominant_peak_nm,Bottom_centroid_nm,Top_centroid_nm,Bottom_log10_integrated_pl,Top_log10_integrated_pl,Abs_signal_450nm,Abs_signal_650nm,Abs_signal_750nm,Top_minus_Bottom_peak_nm,Top_minus_Bottom_centroid_nm
0,A1,A,1,0.900,0.1,0.0,0.000,1.0000,0.0000,0.0,...,796.0,798.670772,793.513213,5.074690,4.641999,1.137,0.784,0.638,-8.0,-5.157559
1,A2,A,2,0.875,0.1,0.0,0.025,1.0000,0.0000,0.0,...,800.0,799.003212,793.793578,5.127319,4.654614,0.982,0.944,0.818,1.0,-5.209634
2,A3,A,3,0.850,0.1,0.0,0.050,1.0000,0.0000,0.0,...,785.0,786.070694,777.495514,4.930852,4.427226,0.754,0.553,0.471,-4.0,-8.575180
3,A4,A,4,0.800,0.1,0.0,0.100,1.0000,0.0000,0.0,...,777.0,778.954967,771.642733,5.030859,4.551035,0.721,0.490,0.416,-7.0,-7.312234
4,A5,A,5,0.900,0.1,0.0,0.000,0.8335,0.1665,0.0,...,752.0,751.787562,746.566288,5.200939,4.726454,0.987,0.612,0.456,-6.0,-5.221274


In [9]:
# Reusable Challenge 1 composition overview
QUATERNARY_METRICS = {
    "Top_minus_Bottom_peak_nm": "Top - Bottom dominant peak (nm)",
    "Top_minus_Bottom_centroid_nm": "Top - Bottom centroid (nm)",
    "Top_dominant_peak_nm": "Top dominant peak (nm)",
    "Bottom_dominant_peak_nm": "Bottom dominant peak (nm)",
    "Abs_signal_650nm": "Absorbance signal at 650 nm",
}
QUATERNARY_COLORBAR_TITLES = {
    "Top_minus_Bottom_peak_nm": "Δ peak<br>(nm)",
    "Top_minus_Bottom_centroid_nm": "Δ centroid<br>(nm)",
    "Top_dominant_peak_nm": "Top peak<br>(nm)",
    "Bottom_dominant_peak_nm": "Bottom peak<br>(nm)",
    "Abs_signal_650nm": "Absorbance<br>at 650 nm",
}

def quaternary_overview(metric="Top_minus_Bottom_peak_nm"):
    diverging = metric.startswith("Top_minus_Bottom")
    figure = px.scatter(
        quaternary_features,
        x="Rb_design_percent", y="DMA_design_percent", color=metric,
        facet_row="Cs_design_percent", facet_col="Br_design_percent",
        category_orders={
            "Cs_design_percent": sorted(
                quaternary_features["Cs_design_percent"].unique(), reverse=True
            ),
            "Br_design_percent": sorted(
                quaternary_features["Br_design_percent"].unique()
            ),
        },
        custom_data=["well", "FA_fraction", "I_fraction", "Br_fraction"],
        labels={
            "Rb_design_percent": "Rb design level (%)",
            "DMA_design_percent": "DMA design level (%)",
            metric: QUATERNARY_METRICS[metric],
        },
        color_continuous_scale="RdBu_r" if diverging else "Viridis",
        color_continuous_midpoint=0 if diverging else None,
        title=(
            "All 96 compositions"
            f"<br><sup>Color: {QUATERNARY_METRICS[metric]}</sup>"
        ),
        height=760,
    )
    figure.update_traces(
        marker={"size": 16, "line": {"color": "white", "width": 1}},
        hovertemplate=(
            "<b>Well %{customdata[0]}</b><br>Rb %{x:g}%<br>DMA %{y:g}%<br>"
            "FA %{customdata[1]:.3f}<br>I %{customdata[2]:.3f}<br>"
            "Br %{customdata[3]:.3f}<br>Value %{marker.color:.4g}<extra></extra>"
        ),
    )
    figure.for_each_annotation(
        lambda annotation: annotation.update(
            text=annotation.text
            .replace("Cs_design_percent=", "Cs = ")
            .replace("Br_design_percent=", "Br = ")
            + "%"
        )
    )
    figure.update_coloraxes(colorbar={
        "title": {
            "text": QUATERNARY_COLORBAR_TITLES[metric], "side": "right"
        },
        "thickness": 18,
        "len": 0.72,
        "x": 1.02,
    })
    figure.update_xaxes(automargin=True)
    figure.update_yaxes(automargin=True)
    figure.update_layout(
        template="plotly_white", autosize=True,
        margin={"l": 75, "r": 135, "t": 110, "b": 80},
        title={"x": 0.01, "xanchor": "left"},
    )
    return figure


In [10]:
# Reusable Challenge 1 full-spectrum view
def quaternary_spectra(well="H8"):
    sample = quaternary_optical[quaternary_optical["well"].eq(well)]
    composition = quaternary_composition[
        quaternary_composition["well"].eq(well)
    ].iloc[0]
    figure = make_subplots(
        rows=2, cols=1, vertical_spacing=0.18,
        subplot_titles=("Top and Bottom PL shape", "Absorbance channel"),
    )
    for geometry, color in [("Top", "#2563eb"), ("Bottom", "#f97316")]:
        rows = sample[sample["geometry"].eq(geometry)].sort_values("wavelength_nm")
        wavelength = rows["wavelength_nm"].to_numpy(float)
        raw = rows["optical_signal_instrument_units"].to_numpy(float)
        signal = np.clip(raw - np.nanpercentile(raw, 5), 0, None)
        signal = signal / signal.max()
        figure.add_trace(go.Scatter(
            x=wavelength, y=signal, mode="lines", name=f"{geometry} PL",
            line={"color": color, "width": 2.5},
        ), row=1, col=1)

    absorbance = sample[sample["geometry"].eq("Absorbance")].sort_values("wavelength_nm")
    figure.add_trace(go.Scatter(
        x=absorbance["wavelength_nm"],
        y=absorbance["optical_signal_instrument_units"],
        mode="lines", name="Absorbance", line={"color": "#059669", "width": 2.5},
    ), row=2, col=1)
    subtitle = (
        f"FA {composition.FA_fraction:.3f} | Cs {composition.Cs_fraction:.3f} | "
        f"Rb {composition.Rb_fraction:.3f} | DMA {composition.DMA_fraction:.3f} | "
        f"I {composition.I_fraction:.3f} | Br {composition.Br_fraction:.3f}"
    )
    figure.update_xaxes(title_text="Wavelength (nm)", row=2, col=1)
    figure.update_yaxes(title_text="Peak-normalized PL", row=1, col=1)
    figure.update_yaxes(title_text="Instrument units", row=2, col=1)
    figure.update_layout(
        title=f"Well {well}<br><sup>{subtitle}</sup>", height=760,
        template="plotly_white", hovermode="x unified", autosize=True,
        margin={"l": 85, "r": 45, "t": 105, "b": 75},
        title_x=0.01,
    )
    return figure


# Worked Example — Quaternary System

### Question

Where do Top and Bottom PL have different dominant peaks, and what do the complete
spectra look like at one selected composition?

### Baseline solution

The overview uses small multiples for Cs and Br, Rb/DMA position, and color for the
optical difference. The second figure shows Top PL, Bottom PL, and absorbance for well
H8. Hover retains the well and composition details. The figures are intentionally
stacked so neither is squeezed into half of the notebook width.


In [11]:
QUATERNARY_EXAMPLE_WELL = "H8"
quaternary_example_figures = [
    quaternary_overview("Top_minus_Bottom_peak_nm"),
    quaternary_spectra(QUATERNARY_EXAMPLE_WELL),
]
show_figures(quaternary_example_figures)

record = quaternary_features[
    quaternary_features["well"].eq(QUATERNARY_EXAMPLE_WELL)
].iloc[0]
display(Markdown(f'''
### What did we just do?

At **{QUATERNARY_EXAMPLE_WELL}**, the Top-minus-Bottom
dominant-peak difference is **{record.Top_minus_Bottom_peak_nm:+.0f} nm**, while the
centroid difference is **{record.Top_minus_Bottom_centroid_nm:+.2f} nm**. Those two
summaries can even disagree in sign, which is why the full line shapes remain visible.
This is an optical comparison, not a phase assignment.
'''))



### What did we just do?

At **H8**, the Top-minus-Bottom
dominant-peak difference is **+11 nm**, while the
centroid difference is **-2.89 nm**. Those two
summaries can even disagree in sign, which is why the full line shapes remain visible.
This is an optical comparison, not a phase assignment.


# <font color="red">Challenge 1 — Quaternary System</font>

### Your task

Improve the baseline so a new user can connect the four A-site fractions, halide
level, measurement geometry, and full spectra more quickly or more accurately.

### Possible solutions

- add coordinated selection between the overview and spectrum;
- try tetrahedral slices or a different composition encoding;
- make absorbance context more direct;
- compare several wells or geometries without losing the full line shapes;
- build a wavelength-first map or compact spectral glyph view.

### Scientific question

> What trend can a new user discover in 60 seconds, and which spectrum supports it?

### Minimum deliverable

One readable visualization, one evidence-bearing spectrum, one sentence describing the
insight, and one sentence naming a possible misinterpretation.


In [13]:

# ============================================================
# CHALLENGE 1: Interactive Composition–Spectrum Explorer
# ============================================================

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

CHALLENGE1_METRIC = "Top_minus_Bottom_peak_nm"
CHALLENGE1_WELL = "H8"

data = quaternary_features.copy().sort_values("well")
wells = data["well"].tolist()
records = data.set_index("well")

# Prepare all measured spectra for efficient selection
spectra = {}
for (well, geometry), group in quaternary_optical.groupby(
    ["well", "geometry"]
):
    group = group.sort_values("wavelength_nm")
    x = group["wavelength_nm"].to_numpy(dtype=float)
    y = group["optical_signal_instrument_units"].to_numpy(
        dtype=float
    )

    if geometry in ("Top", "Bottom"):
        # Same normalization method as the notebook baseline
        y = np.clip(y - np.nanpercentile(y, 5), 0, None)
        maximum = np.nanmax(y)
        y = y / maximum if maximum > 0 else np.zeros_like(y)

    spectra[(well, geometry)] = (x, y)

# Br composition is represented by marker shape
br_levels = sorted(data["Br_design_percent"].unique())
symbols = ["circle", "square", "diamond", "triangle-up",
           "cross", "star"]
symbol_map = {
    level: symbols[i % len(symbols)]
    for i, level in enumerate(br_levels)
}

# Cs composition is represented by marker size
cs = data["Cs_fraction"].to_numpy(dtype=float)
cs_range = np.ptp(cs)
marker_sizes = (
    11 + 19 * (cs - cs.min()) / cs_range
    if cs_range > 0 else np.full(len(cs), 17)
)

delta = data[CHALLENGE1_METRIC].to_numpy(dtype=float)
color_limit = max(np.nanmax(np.abs(delta)), 1)

fig = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{"rowspan": 2}, {}],
        [None, {}]
    ],
    column_widths=[0.53, 0.47],
    vertical_spacing=0.20,
    horizontal_spacing=0.12,
    subplot_titles=[
        "96-well composition overview",
        "Top vs Bottom PL",
        "Absorbance spectrum"
    ]
)

# TRACE 0: All compositions
fig.add_trace(
    go.Scatter(
        x=data["Rb_fraction"],
        y=data["DMA_fraction"],
        mode="markers",
        name="All compositions",
        customdata=data[
            ["well", "FA_fraction", "Cs_fraction",
             "Rb_fraction", "DMA_fraction",
             "I_fraction", "Br_fraction", "Cl_fraction"]
        ].to_numpy(),
        marker=dict(
            size=marker_sizes,
            symbol=[
                symbol_map[v]
                for v in data["Br_design_percent"]
            ],
            color=delta,
            colorscale="RdBu_r",
            cmin=-color_limit,
            cmax=color_limit,
            showscale=True,
            colorbar=dict(
                title="Top − Bottom<br>peak (nm)",
                len=0.65,
                thickness=13,
                x=0.43
            ),
            line=dict(color="white", width=1)
        ),
        hovertemplate=(
            "<b>Well %{customdata[0]}</b><br>"
            "FA: %{customdata[1]:.3f}<br>"
            "Cs: %{customdata[2]:.3f}<br>"
            "Rb: %{customdata[3]:.3f}<br>"
            "DMA: %{customdata[4]:.3f}<br>"
            "I: %{customdata[5]:.3f}<br>"
            "Br: %{customdata[6]:.3f}<br>"
            "Cl: %{customdata[7]:.3f}<br>"
            "Peak difference: %{marker.color:+.1f} nm"
            "<extra></extra>"
        )
    ),
    row=1, col=1
)

# TRACE 1: Highlight selected well
selected = records.loc[CHALLENGE1_WELL]

fig.add_trace(
    go.Scatter(
        x=[selected["Rb_fraction"]],
        y=[selected["DMA_fraction"]],
        mode="markers+text",
        text=[CHALLENGE1_WELL],
        textposition="top center",
        name="Selected well",
        marker=dict(
            size=27,
            symbol="circle-open",
            color="black",
            line=dict(width=3)
        ),
        hoverinfo="skip"
    ),
    row=1, col=1
)

# TRACES 2, 3, 4: Full measured curves
for geometry, color, row in [
    ("Top", "#2563eb", 1),
    ("Bottom", "#f97316", 1),
    ("Absorbance", "#059669", 2)
]:
    x, y = spectra[(CHALLENGE1_WELL, geometry)]
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="lines",
            name=geometry,
            line=dict(color=color, width=2.5)
        ),
        row=row, col=2
    )

# Dropdown: selecting a well updates the highlighted point,
# the three complete spectra, and the title
buttons = []

for well in wells:
    r = records.loc[well]
    top_x, top_y = spectra[(well, "Top")]
    bot_x, bot_y = spectra[(well, "Bottom")]
    abs_x, abs_y = spectra[(well, "Absorbance")]

    title = (
        f"Selected well: {well} | "
        f"Top − Bottom peak: "
        f"{r[CHALLENGE1_METRIC]:+.1f} nm"
        "<br><sup>"
        f"FA={r['FA_fraction']:.2f}, "
        f"Cs={r['Cs_fraction']:.2f}, "
        f"Rb={r['Rb_fraction']:.2f}, "
        f"DMA={r['DMA_fraction']:.2f} | "
        f"I={r['I_fraction']:.2f}, "
        f"Br={r['Br_fraction']:.2f}, "
        f"Cl={r['Cl_fraction']:.2f}"
        "</sup>"
    )

    buttons.append(
        dict(
            label=well,
            method="update",
            args=[
                {
                    "x": [
                        [r["Rb_fraction"]],
                        top_x, bot_x, abs_x
                    ],
                    "y": [
                        [r["DMA_fraction"]],
                        top_y, bot_y, abs_y
                    ]
                },
                {"title.text": title},
                [1, 2, 3, 4]
            ]
        )
    )

initial_title = next(
    b["args"][1]["title.text"]
    for b in buttons
    if b["label"] == CHALLENGE1_WELL
)

fig.update_layout(
    title=dict(text=initial_title, x=0.02, y=0.98),
    template="plotly_white",
    height=790,
    autosize=True,
    hovermode="closest",
    margin=dict(l=75, r=45, t=165, b=80),
    legend=dict(
        orientation="h",
        x=0.55, y=-0.10
    ),
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            active=wells.index(CHALLENGE1_WELL),
            x=0.76, y=1.08,
            xanchor="left",
            yanchor="bottom",
            showactive=True,
            bgcolor="white"
        )
    ],
    annotations=[
        dict(
            text="SELECT WELL",
            x=0.64, y=1.11,
            xref="paper", yref="paper",
            showarrow=False
        )
    ]
)

fig.update_xaxes(
    title_text="Rb fraction", row=1, col=1
)
fig.update_yaxes(
    title_text="DMA fraction", row=1, col=1
)
fig.update_xaxes(
    title_text="Wavelength (nm)", row=1, col=2
)
fig.update_xaxes(
    title_text="Wavelength (nm)", row=2, col=2
)
fig.update_yaxes(
    title_text="Peak-normalized PL", row=1, col=2
)
fig.update_yaxes(
    title_text="Absorbance (instrument units)",
    row=2, col=2
)

# Save figure in the expected notebook variable
challenge1_figures = [fig]

# Display
show_figures(challenge1_figures)

# Automatically report a measured example
r = records.loc[CHALLENGE1_WELL]
display(Markdown(
    f"**Evidence — Well {CHALLENGE1_WELL}:** "
    f"Top PL peak = {r['Top_dominant_peak_nm']:.1f} nm; "
    f"Bottom PL peak = {r['Bottom_dominant_peak_nm']:.1f} nm; "
    f"difference = {r[CHALLENGE1_METRIC]:+.1f} nm."
))



**Evidence — Well H8:** Top PL peak = 752.0 nm; Bottom PL peak = 741.0 nm; difference = +11.0 nm.

<h2><font color="purple">3.</font> Explore the RP-Additive System</h2>

### Problem 2

> **Where do emission bands appear, disappear, or shift with composition and time,
> and do Top and Bottom measurements tell the same story?**

Columns vary the nominal 3AMP/FAPbI3 host mix. Rows vary the nominal MACl design
level. Each well has Top and Bottom PL every 20 minutes from 0 to 380 minutes.

The host fractions form one closed basis. MACl is a separate experimental axis. A
single final-time map hides the trajectory, while 20 separate maps are cumbersome.
The baseline solution pairs a **composition overview at one time** with a
**time × wavelength heatmap** for one selected well.


In [14]:
# Reusable Challenge 2 composition map at one elapsed time
RP_METRICS = {
    "red_fraction_760_820": "Fraction of PL from 760–820 nm",
    "peak_nm": "Dominant emission (nm)",
    "peak_shift_vs_t0": "Peak shift from t0 (nm)",
    "retention_vs_t0": "Integrated PL / t0",
}
RP_COLORBAR_TITLES = {
    "red_fraction_760_820": "760–820 nm<br>PL fraction",
    "peak_nm": "Peak<br>(nm)",
    "peak_shift_vs_t0": "Peak shift<br>(nm)",
    "retention_vs_t0": "PL / t0",
}

def rp_composition_map(
    measure="Bottom PL", time_min=380, metric="red_fraction_760_820",
    selected_well="H7",
):
    current = rp_features[
        rp_features["measure"].eq(measure)
        & np.isclose(rp_features["time_min"], float(time_min))
    ].copy()
    valid = current[current["qc_status"].eq("valid")]
    all_valid = rp_features[
        rp_features["measure"].eq(measure)
        & rp_features["qc_status"].eq("valid")
    ]
    diverging = metric == "peak_shift_vs_t0"
    if diverging:
        bound = float(np.nanmax(np.abs(all_valid[metric])))
        color_range = [-bound, bound]
    else:
        color_range = [float(all_valid[metric].min()), float(all_valid[metric].max())]

    figure = px.scatter(
        valid, x="FAPbI3_percent", y="additive_percent", color=metric,
        custom_data=["well", "spacer_percent", "qc_status"],
        labels={
            "FAPbI3_percent": "FAPbI3 host (%)",
            "additive_percent": "MACl level (%)",
            metric: RP_METRICS[metric],
        },
        color_continuous_scale="RdBu_r" if diverging else "Viridis",
        range_color=color_range,
        title=(
            f"{measure} at {time_min:g} min"
            f"<br><sup>Color: {RP_METRICS[metric]}</sup>"
        ),
        height=650,
    )
    figure.update_traces(
        marker={"size": 16, "line": {"color": "white", "width": 1}},
        hovertemplate=(
            "<b>Well %{customdata[0]}</b><br>FAPbI3 %{x:.1f}%<br>"
            "3AMP host %{customdata[1]:.1f}%<br>MACl level %{y:g}%<br>"
            "Value %{marker.color:.4g}<extra></extra>"
        ),
    )
    invalid = current[~current["qc_status"].eq("valid")]
    if not invalid.empty:
        figure.add_trace(go.Scatter(
            x=invalid["FAPbI3_percent"], y=invalid["additive_percent"],
            mode="markers", name="Low/invalid PL",
            marker={"size": 13, "symbol": "x", "color": "#6b7280"},
            text=invalid["well"], hovertemplate="Well %{text}<extra></extra>",
        ))
    selected = current[current["well"].eq(selected_well)]
    if not selected.empty:
        figure.add_trace(go.Scatter(
            x=selected["FAPbI3_percent"], y=selected["additive_percent"],
            mode="markers", name=f"Selected: {selected_well}",
            marker={"size": 20, "symbol": "circle-open", "color": "black",
                    "line": {"width": 3}},
            hoverinfo="skip",
        ))
    figure.update_coloraxes(colorbar={
        "title": {"text": RP_COLORBAR_TITLES[metric], "side": "right"},
        "thickness": 18,
        "len": 0.72,
        "x": 1.02,
    })
    figure.update_xaxes(range=[-5, 105], automargin=True)
    figure.update_yaxes(range=[-1, 21.5], automargin=True)
    figure.update_layout(
        template="plotly_white", autosize=True,
        margin={"l": 80, "r": 145, "t": 105, "b": 120},
        title={"x": 0.01, "xanchor": "left"},
        legend={
            "orientation": "h", "yanchor": "top", "y": -0.20,
            "xanchor": "left", "x": 0,
        },
    )
    return figure


In [15]:
# Reusable Challenge 2 time × wavelength view for one composition
def rp_time_wavelength(well="H7", measure="Bottom PL"):
    rows = rp_features[
        rp_features["well"].eq(well) & rp_features["measure"].eq(measure)
    ].sort_values("time_min")
    matrix = rp_spectra_by_id.loc[
        rows["spectrum_id"], rp_wavelength_columns
    ].to_numpy(float, copy=True)
    valid = rows["qc_status"].eq("valid").to_numpy()
    matrix[~valid] = np.nan
    maxima = np.ones((len(rows), 1))
    maxima[valid] = np.max(matrix[valid], axis=1, keepdims=True)
    matrix = matrix / maxima

    composition = rp_composition[rp_composition["well"].eq(well)].iloc[0]
    subtitle = (
        f"FAPbI3 host {composition.FAPbI3_percent:.1f}% | "
        f"3AMP host {composition.spacer_percent:.1f}% | "
        f"MACl nominal level {composition.additive_percent:g}%"
    )
    figure = go.Figure(go.Heatmap(
        x=rp_wavelengths, y=rows["time_min"], z=matrix,
        zmin=0, zmax=1, colorscale="Magma",
        colorbar={
            "title": {"text": "PL shape<br>(0–1)", "side": "right"},
            "thickness": 18,
            "len": 0.78,
            "x": 1.02,
        },
        hovertemplate="%{x:.0f} nm<br>%{y:.0f} min<br>%{z:.3f}<extra></extra>",
    ))
    figure.update_layout(
        title=f"Well {well} | {measure}<br><sup>{subtitle}</sup>",
        xaxis_title="Wavelength (nm)", yaxis_title="Elapsed time (min)",
        height=650, template="plotly_white", plot_bgcolor="#d1d5db",
        autosize=True, margin={"l": 85, "r": 125, "t": 105, "b": 75},
        title_x=0.01,
    )
    return figure


# Worked Example — RP-Additive System

### Question

Which compositions show strong long-wavelength PL at 380 minutes, and how did the
spectrum of one selected well evolve to that state?

### Baseline solution

The map uses fixed color limits across time. Low/invalid PL is gray rather than being
silently converted to zero. The second figure retains the complete time–wavelength
history for well H7. The figures are stacked at full width for legibility.


In [16]:
RP_EXAMPLE_WELL = "H7"
RP_EXAMPLE_MEASURE = "Bottom PL"
rp_example_figures = [
    rp_composition_map(
        RP_EXAMPLE_MEASURE, 380, "red_fraction_760_820", RP_EXAMPLE_WELL
    ),
    rp_time_wavelength(RP_EXAMPLE_WELL, RP_EXAMPLE_MEASURE),
]
show_figures(rp_example_figures)

history = rp_features[
    rp_features["well"].eq(RP_EXAMPLE_WELL)
    & rp_features["measure"].eq(RP_EXAMPLE_MEASURE)
].sort_values("time_min")
initial, final = history.iloc[0], history.iloc[-1]
display(Markdown(f'''
### What did we just do?

At **{RP_EXAMPLE_WELL}**, the Bottom-PL dominant maximum
changes from **{initial.peak_nm:.0f} nm** at {initial.time_min:.0f} minutes to
**{final.peak_nm:.0f} nm** at {final.time_min:.0f} minutes. Its 760–820 nm fraction
changes from **{initial.red_fraction_760_820:.3f}** to
**{final.red_fraction_760_820:.3f}**. The heatmap shows whether that scalar change is
a moving band, a competing band, or a change in spectral weight. It is not by itself
evidence of a structural phase transition.
'''))



### What did we just do?

At **H7**, the Bottom-PL dominant maximum
changes from **706 nm** at 0 minutes to
**772 nm** at 380 minutes. Its 760–820 nm fraction
changes from **0.163** to
**0.388**. The heatmap shows whether that scalar change is
a moving band, a competing band, or a change in spectral weight. It is not by itself
evidence of a structural phase transition.


# <font color="red">Challenge 2 — RP-Additive System</font>

### Your task

Improve the baseline so a new user can connect host composition, MACl design level,
time, wavelength, and measurement geometry more quickly or more accurately.

### Possible solutions

- add a time control or animation with a fixed color scale;
- compare Top and Bottom measurements directly;
- encode final − initial change or show a trajectory through time;
- create a wavelength-first composition map;
- link a composition selection to its full spectral history.

### Scientific question

> Where does a spectral feature change, when does it change, and what composition
> context makes that trend understandable?

### Minimum deliverable

One readable visualization, one time-resolved spectral view, one sentence describing
the insight, and one sentence naming a possible misinterpretation.


In [18]:
# ============================================================
# YOUR CODE STARTS HERE — Challenge 2
# ============================================================

CHALLENGE2_MEASURE = "Bottom PL"
CHALLENGE2_TIME_MIN = 380
CHALLENGE2_METRIC = "red_fraction_760_820"
CHALLENGE2_WELL = "H7"

# Start from this working baseline, then replace or extend it.
challenge2_figures = [
    rp_composition_map(
        CHALLENGE2_MEASURE, CHALLENGE2_TIME_MIN,
        CHALLENGE2_METRIC, CHALLENGE2_WELL,
    ),
    rp_time_wavelength(CHALLENGE2_WELL, CHALLENGE2_MEASURE),
]

# Ideas: compare geometries, add a time control, encode two metrics,
# or replace both panels with one more integrated representation.

# ============================================================
# YOUR CODE ENDS HERE
# ============================================================
show_figures(challenge2_figures)


<h2><font color="purple">4.</font> Build, Test, and Explain Your Design</h2>

Use a **60-second usability test**: hand the visualization to someone from another team
and ask them to answer your scientific question without coaching.

Score one point for each:

1. **Dimensional fidelity:** important variables are shown, encoded, or selectable.
2. **Overview + evidence:** a trend can be found and its spectra inspected.
3. **Spectral honesty:** summaries do not masquerade as phase assignments.
4. **Comparable scales + QC:** time/geometry comparisons use defensible scales, and
   low/invalid PL is visibly distinct from measured zero.
5. **One-minute usability:** a new user can answer the stated question.


In [19]:
# @title 5. Team result — edit these three fields
TEAM_NAME = "" # @param {type:"string"}
CHOSEN_CHALLENGE = "Challenge 1 — Quaternary System" # @param ["Challenge 1 — Quaternary System", "Challenge 2 — RP-Additive System"]
DESIGN_CLAIM = "" # @param {type:"string"}

TEAM_FIGURES = (
    challenge1_figures
    if CHOSEN_CHALLENGE.startswith("Challenge 1")
    else challenge2_figures
)
display(Markdown(
    f"### {TEAM_NAME or 'Unnamed team'}\n\n"
    f"**{CHOSEN_CHALLENGE}**\n\n"
    f"{DESIGN_CLAIM or 'Add one sentence explaining what became easier to see.'}"
))
show_figures(TEAM_FIGURES)


### Unnamed team

**Challenge 1 — Quaternary System**

Add one sentence explaining what became easier to see.

In [20]:
# @title 6. Export the final interactive result
safe_team = "".join(
    character if character.isalnum() or character in "-_" else "_"
    for character in (TEAM_NAME.strip() or "unnamed_team")
)
export_dir = DATA_ROOT / "team_exports"
export_dir.mkdir(exist_ok=True)
export_path = export_figure_grid(
    TEAM_FIGURES,
    export_dir / f"{safe_team}_final_visualization.html",
    f"{TEAM_NAME or 'Unnamed team'} — {CHOSEN_CHALLENGE}",
)
print("Wrote", export_path)
display(FileLink(str(export_path), result_html_prefix="Download interactive HTML: "))


Wrote /content/higher_dimensional_optical_data/team_exports/unnamed_team_final_visualization.html


/content/higher_dimensional_optical_data/team_exports/unnamed_team_final_visualization.html

## Final Discussion

Be ready to answer:

1. What became easier to see?
2. Which dimensions are directly shown, encoded, selected, or omitted?
3. What could the design cause someone to assume incorrectly?
4. Does the conclusion still hold when the full spectra are inspected?
5. What would you change with another hour?

---

## <font color="green">Takeaway</font>

A useful higher-dimensional visualization does not merely make the plot look compact.
It preserves the scientific variables, exposes comparison choices, distinguishes bad
signal from measured zero, and keeps the underlying spectra inspectable.


## Optional Appendix — Data Provenance and Limits

**Quaternary System:** 96 static wells; one Top PL, Bottom PL, and absorbance read per well.
Nominal composition fractions assume equal-molar stocks. Stock concentrations and
solvents are not included.

**RP-Additive System:** 96 wells × 20 times × two geometries = 3,840 PL spectra. The MACl
percentage is a nominal design label with undocumented denominator. The dispense table
was reconstructed from project plate-layout code and cross-checked against all 96
metadata labels; it was not parsed from the plate-reader workbook.

Each downloaded challenge folder includes `README_DATA.md` and `SHA256SUMS.txt` for a
fuller field description and file-integrity record.


In [21]:
"""
Higher-Dimensional Optical Data Visualization Challenge - complete solution in ONE file.

Solves both challenges of the ATHENA-CAMM hackathon notebook
"Higher_Dimensional_Optical_Data_Hackathon.ipynb":

    Challenge 1 - Quaternary System   (96 wells: Top PL, Bottom PL, absorbance)
    Challenge 2 - RP-Additive System  (96 wells x 20 times: Top PL, Bottom PL)

HOW TO RUN
    Google Colab / Jupyter : paste this whole file into one cell and run it.
                             The figures appear below the cell.
    Terminal               : python hd_optical_hackathon_solution.py
                             Then open the two .html files it writes.

WHAT IT DOES, IN ORDER
    1. Settings                      (edit TEAM_NAME and CHOSEN_CHALLENGE here)
    2. Imports
    3. Download + checksum the data  (same code as the notebook)
    4. Load the tables and run the structure checks
    5. Display / export helpers      (same code as the notebook)
    6. Baseline Quaternary features  (same code as the notebook)
    7. Solution helper: band_by_area
    8. Challenge 1 solution          -> challenge1_figures + printed summary
    9. Challenge 2 solution          -> challenge2_figures + printed summary
   10. Export interactive HTML files

Needs: numpy, pandas, plotly, pyarrow, gdown (all preinstalled on Colab).
"""

# ============================================================================
# 1. SETTINGS - the only lines you normally need to edit
# ============================================================================
TEAM_NAME = ""                 # your team name; used in the page title and file name
CHOSEN_CHALLENGE = "both"      # "1", "2" or "both": which solution(s) to show and export
OUTPUT_FOLDER = "hd_optical_outputs"   # the .html files are written here


# ============================================================================
# 2. IMPORTS
# ============================================================================
import importlib
import subprocess
import sys

# Install anything that is missing (does nothing on Colab, where all five exist).
for _module, _package in [("gdown", "gdown"), ("pyarrow", "pyarrow>=14"), ("plotly", "plotly>=5.22"),
                          ("pandas", "pandas"), ("numpy", "numpy")]:
    try:
        importlib.import_module(_module)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _package])

from html import escape
from pathlib import Path
import hashlib
import shutil

import gdown
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Works both inside a notebook (rich output) and in a plain terminal (printed text).
try:
    from IPython import get_ipython
    from IPython.display import HTML, Markdown, display
    _shell = get_ipython()
    IN_NOTEBOOK = _shell is not None and _shell.__class__.__name__ != "TerminalInteractiveShell"
except Exception:
    IN_NOTEBOOK = False
if not IN_NOTEBOOK:
    def Markdown(text):
        return text

    def HTML(text):
        return text

    def display(thing):
        print(thing.to_string() if hasattr(thing, "to_string") else thing)


# ============================================================================
# 3. DOWNLOAD BOTH DATA FOLDERS AND VERIFY THEIR CHECKSUMS
# ============================================================================

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/16YfPjFdZHgW3hkC0QBLrCS9um2C6aK5q?usp=drive_link"
IN_COLAB = "google.colab" in sys.modules
DATA_ROOT = Path(
    "/content/higher_dimensional_optical_data"
    if IN_COLAB else "./.higher_dimensional_optical_data"
).resolve()
STAGING_ROOT = DATA_ROOT.with_name(DATA_ROOT.name + "_download")

REMOTE_QUATERNARY_FOLDER = "01_hochan_quaternary_data"
EXPECTED_MANIFEST_HASHES = {
    REMOTE_QUATERNARY_FOLDER: "6c4f5329411f7897b16578ec3f029f092ca05eb99a884a9a622b1ffabf24ff8e",
    "02_rp_additive_data": "0060b641f417179122562eefb6d5acee527b4927796fd4a595631b683fded38c",
}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def folder_errors(root):
    errors = []
    for folder_name, expected_manifest_hash in EXPECTED_MANIFEST_HASHES.items():
        folder = root / folder_name
        checksum_path = folder / "SHA256SUMS.txt"
        if not checksum_path.is_file():
            errors.append(f"missing: {folder_name}/SHA256SUMS.txt")
            continue
        if file_sha256(checksum_path) != expected_manifest_hash:
            errors.append(f"unexpected checksum manifest: {folder_name}")
            continue
        for line in checksum_path.read_text(encoding="utf-8").strip().splitlines():
            expected, relative = line.split("  ", 1)
            path = folder / relative
            if not path.is_file():
                errors.append(f"missing: {folder_name}/{relative}")
            elif file_sha256(path) != expected:
                errors.append(f"checksum mismatch: {folder_name}/{relative}")
    return errors

def locate_download_root(root):
    candidates = [root] + sorted(
        (path for path in root.rglob("*") if path.is_dir()),
        key=lambda path: len(path.parts),
    )
    matches = [
        path for path in candidates
        if (path / REMOTE_QUATERNARY_FOLDER).is_dir()
        and (path / "02_rp_additive_data").is_dir()
    ]
    if len(matches) != 1:
        raise RuntimeError(
            "Expected one downloaded parent containing both challenge folders; "
            f"found {len(matches)}."
        )
    return matches[0]

errors = folder_errors(DATA_ROOT)
if errors:
    if STAGING_ROOT.exists():
        shutil.rmtree(STAGING_ROOT)
    STAGING_ROOT.mkdir(parents=True)
    print("Downloading both challenge datasets from Google Drive ...")
    try:
        downloaded = gdown.download_folder(
            url=DRIVE_FOLDER_URL,
            output=str(STAGING_ROOT),
            quiet=False,
            use_cookies=False,
        )
        if not downloaded:
            raise RuntimeError("Google Drive returned no files.")
        downloaded_root = locate_download_root(STAGING_ROOT)
        errors = folder_errors(downloaded_root)
        if errors:
            raise RuntimeError("Downloaded data failed verification:\n- " + "\n- ".join(errors))

        if DATA_ROOT.exists():
            shutil.rmtree(DATA_ROOT)
        if downloaded_root == STAGING_ROOT:
            STAGING_ROOT.rename(DATA_ROOT)
        else:
            shutil.move(str(downloaded_root), str(DATA_ROOT))
            shutil.rmtree(STAGING_ROOT, ignore_errors=True)
    except Exception as exc:
        shutil.rmtree(STAGING_ROOT, ignore_errors=True)
        raise RuntimeError(
            "Data setup failed. Confirm that the shared parent folder and its contents "
            "are set to 'Anyone with the link - Viewer', then rerun this cell."
        ) from exc
    print("Download complete; every file passed SHA-256 verification.")
else:
    print("Verified data already present; no download needed.")

QUATERNARY_DIR = DATA_ROOT / REMOTE_QUATERNARY_FOLDER
RP_DIR = DATA_ROOT / "02_rp_additive_data"
print("Challenge 1 — Quaternary System: ready")
print("Challenge 2 — RP-Additive System: ready")


# ============================================================================
# 4. LOAD THE TABLES AND RUN THE STRUCTURE CHECKS
# ============================================================================

QUATERNARY_COMPOSITION_PATH = QUATERNARY_DIR / "composition_and_recipe.csv"
RP_COMPOSITION_PATH = RP_DIR / "composition_and_recipe.csv"

quaternary_composition = pd.read_csv(QUATERNARY_COMPOSITION_PATH)
quaternary_optical = pd.read_parquet(QUATERNARY_DIR / "optical_spectra_long.parquet")
quaternary_manifest = pd.read_csv(QUATERNARY_DIR / "measurement_manifest.csv")

rp_composition = pd.read_csv(RP_COMPOSITION_PATH)
rp_features = pd.read_parquet(RP_DIR / "metadata_features.parquet")
rp_spectra = pd.read_parquet(RP_DIR / "spectra_wide.parquet")

rp_wavelength_columns = sorted(
    (column for column in rp_spectra if column.startswith("wl_")),
    key=lambda column: float(column.split("_", 1)[1]),
)
rp_wavelengths = np.asarray([
    float(column.split("_", 1)[1]) for column in rp_wavelength_columns
])
rp_spectra_by_id = rp_spectra.set_index("spectrum_id")

assert len(quaternary_composition) == quaternary_composition["well"].nunique() == 96
assert set(quaternary_composition["well"]) == set(quaternary_optical["well"])
assert set(quaternary_optical["geometry"]) == {"Top", "Bottom", "Absorbance"}

a_site = ["FA_fraction", "Cs_fraction", "Rb_fraction", "DMA_fraction"]
x_site = ["I_fraction", "Br_fraction", "Cl_fraction"]
assert np.allclose(quaternary_composition[a_site].sum(axis=1), 1.0)
assert np.allclose(quaternary_composition[x_site].sum(axis=1), 1.0)

assert len(rp_composition) == rp_composition["well"].nunique() == 96
assert set(rp_composition["well"]) == set(rp_features["well"])
assert len(rp_features) == rp_features["spectrum_id"].nunique() == 3840
assert set(rp_features["spectrum_id"]) == set(rp_spectra["spectrum_id"])
assert np.allclose(
    rp_composition["spacer_host_fraction"]
    + rp_composition["FAPbI3_host_fraction"],
    1.0,
)

coverage = pd.DataFrame([
    {
        "challenge": "Challenge 1 — Quaternary System",
        "wells": quaternary_composition["well"].nunique(),
        "spectra": quaternary_optical.groupby(["well", "geometry"]).ngroups,
        "times": "none",
        "measurements": "Top PL, Bottom PL, absorbance",
    },
    {
        "challenge": "Challenge 2 — RP-Additive System",
        "wells": rp_composition["well"].nunique(),
        "spectra": len(rp_features),
        "times": rp_features["time_min"].nunique(),
        "measurements": "Top PL, Bottom PL",
    },
])
display(Markdown("### Data ready — all integrity and structure checks passed"))
display(coverage)


# ============================================================================
# 5. DISPLAY / EXPORT HELPERS
# ============================================================================

# Responsive display/export helpers used by both worked examples.
# One column is the readable Colab default; use columns=2 only on a wide display.
def show_figures(figures, columns=1):
    if columns not in (1, 2):
        raise ValueError("columns must be 1 or 2")
    blocks = [
        "<div style='min-width:0;width:100%;overflow-x:auto'>"
        + pio.to_html(
            figure,
            include_plotlyjs="cdn" if index == 0 else False,
            full_html=False,
            config={"responsive": True, "displaylogo": False},
        )
        + "</div>"
        for index, figure in enumerate(figures)
    ]
    grid_class = f"hd-figure-grid-{columns}"
    display(HTML(
        f"<style>.{grid_class}{{display:grid;grid-template-columns:repeat("
        f"{columns},minmax(0,1fr));gap:28px;align-items:start;width:100%;"
        "max-width:1200px;margin:0 auto}}"
        f"@media(max-width:1100px){{.{grid_class}{{grid-template-columns:1fr}}}}"
        "</style>"
        f"<div class='{grid_class}'>" + "".join(blocks) + "</div>"
    ))

def export_figure_grid(figures, path, title, columns=1):
    if columns not in (1, 2):
        raise ValueError("columns must be 1 or 2")
    blocks = [
        pio.to_html(
            figure,
            include_plotlyjs=True if index == 0 else False,
            full_html=False,
            config={"responsive": True, "displaylogo": False},
        )
        for index, figure in enumerate(figures)
    ]
    document = (
        "<!doctype html><meta charset='utf-8'><title>" + escape(title) + "</title>"
        "<style>body{font-family:Arial;margin:24px;color:#172b4d}"
        ".grid{display:grid;gap:28px;max-width:1200px;margin:0 auto}"
        f".grid{{grid-template-columns:repeat({columns},minmax(0,1fr))}}"
        "@media(max-width:1100px){.grid{grid-template-columns:1fr}}"
        ".grid>div{min-width:0;width:100%;overflow-x:auto}</style>"
        "<h1>" + escape(title) + "</h1><div class='grid'>"
        + "".join("<div>" + block + "</div>" for block in blocks)
        + "</div>"
    )
    Path(path).write_text(document, encoding="utf-8")
    return Path(path)


# ============================================================================
# 6. BASELINE QUATERNARY FEATURES  (notebook feature cells, unchanged)
#    Gives `quaternary_features`: one row per well, including the baseline's single-point peaks.
# ============================================================================

# Summarize each Quaternary System PL spectrum without changing the source data
integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz

def summarize_pl(group):
    ordered = group.sort_values("wavelength_nm")
    wavelength = ordered["wavelength_nm"].to_numpy(float)
    raw = ordered["optical_signal_instrument_units"].to_numpy(float)
    signal = np.clip(raw - np.nanpercentile(raw, 5), 0, None)
    area = float(integrate(signal, wavelength))
    peak = float(wavelength[int(np.nanargmax(signal))])
    centroid = float(np.sum(wavelength * signal) / signal.sum())
    return pd.Series({
        "dominant_peak_nm": peak,
        "centroid_nm": centroid,
        "log10_integrated_pl": np.log10(area),
    })

quaternary_pl_features = (
    quaternary_optical[quaternary_optical["modality"].eq("PL")]
    .groupby(["well", "geometry"], sort=False)[
        ["wavelength_nm", "optical_signal_instrument_units"]
    ]
    .apply(summarize_pl)
    .reset_index()
)

# Assemble one row per well for composition-space visualization
pl_wide = quaternary_pl_features.pivot(
    index="well", columns="geometry",
    values=["dominant_peak_nm", "centroid_nm", "log10_integrated_pl"],
)
pl_wide.columns = [
    f"{geometry}_{metric}" for metric, geometry in pl_wide.columns
]
pl_wide = pl_wide.reset_index()

absorbance_points = (
    quaternary_optical[
        quaternary_optical["modality"].eq("Absorbance")
        & quaternary_optical["wavelength_nm"].isin([450, 650, 750])
    ]
    .pivot(index="well", columns="wavelength_nm",
           values="optical_signal_instrument_units")
    .rename(columns=lambda value: f"Abs_signal_{int(value)}nm")
    .reset_index()
)

quaternary_features = (
    quaternary_composition.merge(pl_wide, on="well", validate="one_to_one")
    .merge(absorbance_points, on="well", validate="one_to_one")
)
quaternary_features["Top_minus_Bottom_peak_nm"] = (
    quaternary_features["Top_dominant_peak_nm"]
    - quaternary_features["Bottom_dominant_peak_nm"]
)
quaternary_features["Top_minus_Bottom_centroid_nm"] = (
    quaternary_features["Top_centroid_nm"]
    - quaternary_features["Bottom_centroid_nm"]
)


# ============================================================================
# 7. SOLUTION HELPER  (new)
# ============================================================================

# Solution helper — used by BOTH challenge solutions below (run this cell first).
#
# Why it exists: the baseline takes the single highest point of a spectrum as "the peak".
# One noisy point (for example at the edge of the detector range) can then win. This helper
# describes the emission band by where its LIGHT is: the wavelength that splits the band's
# area in half. Area is built from many points, so a single point cannot move the result.
# Nothing is smoothed or fitted, and the raw spectra are never modified.

def coarse_bins(wavelength, signal, bin_nm=10.0):
    """Average the signal in consecutive bins of `bin_nm`. Returns bin edges and bin means.

    Ten-nanometre averages are steady enough to say where a band starts and stops, and no
    single point can dominate one.
    """
    wavelength = np.asarray(wavelength, float)
    signal = np.asarray(signal, float)
    edges = np.arange(wavelength[0], wavelength[-1] + bin_nm, bin_nm)
    index = np.clip(np.digitize(wavelength, edges) - 1, 0, len(edges) - 2)
    counts = np.bincount(index, minlength=len(edges) - 1)
    means = np.bincount(index, weights=signal, minlength=len(edges) - 1) / np.maximum(counts, 1)
    filled = counts > 0
    return edges[:-1][filled], edges[1:][filled], means[filled]


def band_by_area(wavelength, signal, window, bin_nm=10.0, floor=0.2):
    """Summarise the dominant PL band inside `window` (nm) by its area quantiles.

    The band is the unbroken run of 10 nm bins around the brightest bin whose averages stay
    above `floor` (20 %) of that bin. Inside that run, on the measured points themselves:

      centre_nm  wavelength with half of the band's light on each side (area median)
      width_nm   width of the central half of the light (25 % to 75 % of the area)
      lower_nm / upper_nm   those 25 % and 75 % wavelengths
      height     average signal of the brightest bin, above the local baseline
      snr        height / point-to-point noise (how clearly the band stands out)
      clipped    True if the band runs into the window limit (its far tail is cut off)
      at_edge    True if the brightest bin itself is the first or last bin (not a real band)
    """
    wavelength = np.asarray(wavelength, float)
    signal = np.asarray(signal, float)
    keep = (wavelength >= window[0]) & (wavelength <= window[1])
    w, y = wavelength[keep], signal[keep]
    y = y - np.nanpercentile(y, 5)                         # local baseline, as in the starter notebook
    left_edges, right_edges, means = coarse_bins(w, y, bin_nm)
    top = int(np.argmax(means))
    height = float(means[top])
    # point-to-point noise from neighbouring differences (a difference of two points carries
    # the noise of both, hence the division by sqrt(2))
    steps = np.diff(y)
    noise = 1.4826 * float(np.median(np.abs(steps - np.median(steps)))) / np.sqrt(2) + 1e-9

    first = last = top                                     # grow the band outward bin by bin
    while first > 0 and means[first - 1] >= floor * height:
        first -= 1
    while last < len(means) - 1 and means[last + 1] >= floor * height:
        last += 1
    inside = (w >= left_edges[first]) & (w <= right_edges[last])
    band_w, band_y = w[inside], np.clip(y[inside], 0, None)
    share = np.cumsum(band_y)                              # running share of the band's light
    share = share / share[-1] if share[-1] > 0 else np.linspace(0, 1, len(band_w))
    lower, centre, upper = np.interp([0.25, 0.5, 0.75], share, band_w)
    return {
        "centre_nm": float(centre), "width_nm": float(upper - lower),
        "lower_nm": float(lower), "upper_nm": float(upper),
        "height": height, "snr": height / noise,
        "clipped": bool(first == 0 or last == len(means) - 1),
        "at_edge": bool(top == 0 or top == len(means) - 1),
    }


def signed_text(value, digits=0):
    """Format a number with an explicit sign and a real minus sign, e.g. +3 or −1.5."""
    return f"{value:+.{digits}f}".replace("-", "−")


# One colour per measurement, kept identical in every solution figure.
SOLUTION_COLORS = {"Top": "#6250d6", "Bottom": "#1baf7a", "Absorbance": "#eda100"}
SOLUTION_PALE = {"Top": "#cfc9f3", "Bottom": "#b5e6d3"}      # pale version, used for "first read"
SOLUTION_INK = {"text": "#0b0b0b", "muted": "#52514e", "grid": "#e4e3df", "invalid": "#8a8985"}
# Ordered things (design levels, elapsed time) get one hue from light to dark.
SOLUTION_LEVEL_RAMP = ["#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]


# ============================================================================
# 8. CHALLENGE 1 SOLUTION - QUATERNARY SYSTEM  (new)
# ============================================================================

# QUESTION THIS DESIGN ANSWERS
#   How does composition move the PL band, do Top and Bottom agree about it,
#   and does the absorption edge tell the same story?
#
# DESIGN (three stacked figures, all plain Plotly so they survive the HTML export)
#   1. Parallel coordinates - one line per well through every variable at once:
#                             the four composition knobs, then the three measured
#                             wavelengths on ONE shared scale. Drag on any axis to filter.
#   2. Mirror spectra       - every full spectrum: Top drawn upward, Bottom drawn
#                             downward, so agreement shows up as mirror symmetry.
#   3. Cumulative check     - "what share of the wells lies within X nm?", for Top vs Bottom
#                             and for PL band vs absorption edge.

# ---------------------------------------------------------------- settings
Q_BAND_WINDOW = (600.0, 845.0)   # where the PL band is measured; stops before the noisy last few nm
Q_ARTIFACT_NM = 15.0             # baseline single-point peak this far from the band = noise artifact
Q_WEAK_EDGE = 0.5e-3             # absorbance edges shallower than this are treated as "weak"
Q_KNOBS = ["Br", "DMA", "Cs", "Rb"]   # composition knobs, ordered by how much they move the band

# ---------------------------------------------------------------- 1. robust features
q_pl = quaternary_optical[quaternary_optical["modality"].eq("PL")]
q_curves = {}        # (well, geometry) -> wavelength, baseline-subtracted signal, band summary
q_band_rows = []
for (well, geometry), group in q_pl.groupby(["well", "geometry"], sort=False):
    ordered = group.sort_values("wavelength_nm")
    wavelength = ordered["wavelength_nm"].to_numpy(float)
    raw = ordered["optical_signal_instrument_units"].to_numpy(float)
    signal = np.clip(raw - np.nanpercentile(raw, 5), 0, None)
    band = band_by_area(wavelength, raw, Q_BAND_WINDOW)
    q_curves[(well, geometry)] = (wavelength, signal, band)
    q_band_rows.append({"well": well, "geometry": geometry, **band})
q_band = pd.DataFrame(q_band_rows)


def absorbance_edge(wavelength, absorbance, window=(620.0, 840.0), bin_nm=10.0):
    """Wavelength where absorbance falls fastest, compared with its surroundings.

    The curve is averaged in 10 nm bins. The drop from each bin to the next is compared with
    the typical drop nearby (the smooth scattering slope), and the edge is the centre of
    gravity of the unusually large drops around the largest one. Unlike a single absorbance
    value (the baseline uses 650 nm), an edge position does not change when the whole curve
    is offset by scattering or by the plate.
    """
    left_edges, right_edges, means = coarse_bins(wavelength, absorbance, bin_nm)
    drops = means[:-1] - means[1:]                         # how much absorbance falls per bin step
    places = right_edges[:-1]                              # the wavelength between the two bins
    span = 6                                               # compare with +/- 6 bin steps (120 nm)
    typical = np.array([np.median(drops[max(0, i - span):i + span + 1]) for i in range(len(drops))])
    excess = np.where((places >= window[0]) & (places <= window[1]), drops - typical, -np.inf)
    top = int(np.argmax(excess))
    near = slice(max(0, top - 2), top + 3)
    weight = np.clip(excess[near], 0, None)
    weight[~np.isfinite(weight)] = 0
    return float(np.sum(places[near] * weight) / np.sum(weight)), float(excess[top] / bin_nm)


q_abs_curves = {}
q_edge_rows = []
for well, group in quaternary_optical[quaternary_optical["modality"].eq("Absorbance")].groupby("well"):
    ordered = group.sort_values("wavelength_nm")
    wavelength = ordered["wavelength_nm"].to_numpy(float)
    absorbance = ordered["optical_signal_instrument_units"].to_numpy(float)
    edge, depth = absorbance_edge(wavelength, absorbance)
    q_abs_curves[well] = (wavelength, absorbance)
    q_edge_rows.append({"well": well, "abs_edge_nm": edge, "abs_edge_depth": depth})

q_band_wide = q_band.pivot(index="well", columns="geometry",
                           values=["centre_nm", "width_nm", "height", "snr"])
q_band_wide.columns = [f"{geometry}_{name}" for name, geometry in q_band_wide.columns]
q_wells = (
    quaternary_features
    .merge(q_band_wide.reset_index(), on="well", validate="one_to_one")
    .merge(pd.DataFrame(q_edge_rows), on="well", validate="one_to_one")
)
q_wells["delta_centre_nm"] = q_wells["Top_centre_nm"] - q_wells["Bottom_centre_nm"]
q_wells["delta_width_nm"] = q_wells["Top_width_nm"] - q_wells["Bottom_width_nm"]
q_wells["edge_weak"] = q_wells["abs_edge_depth"] < Q_WEAK_EDGE
q_wells["edge_to_pl_nm"] = q_wells["Bottom_centre_nm"] - q_wells["abs_edge_nm"]
# A well is flagged when the baseline's single highest point is far from the real band.
q_wells["naive_artifact"] = (
    (q_wells["Top_dominant_peak_nm"] - q_wells["Top_centre_nm"]).abs().gt(Q_ARTIFACT_NM)
    | (q_wells["Bottom_dominant_peak_nm"] - q_wells["Bottom_centre_nm"]).abs().gt(Q_ARTIFACT_NM)
)

# ---------------------------------------------------------------- shared small helpers
Q_LEVELS = {
    knob: sorted(q_wells[f"{knob}_design_percent"].unique())
    for knob in Q_KNOBS
}


def q_level_text(value):
    return f"{round(float(value), 1):g}"


def q_level_shades(count):
    """`count` ordered shades, light to dark, for ordered design levels."""
    if count == 1:
        return [SOLUTION_LEVEL_RAMP[2]]
    return [SOLUTION_LEVEL_RAMP[int(round(i))] for i in np.linspace(0, 3, count)]


def q_normalised(well, geometry, low=None, high=None):
    """Baseline-subtracted spectrum divided by its band height (so the band top is 1)."""
    wavelength, signal, band = q_curves[(well, geometry)]
    keep = np.ones(len(wavelength), bool)
    if low is not None:
        keep &= wavelength >= low
    if high is not None:
        keep &= wavelength <= high
    return wavelength[keep], np.round(signal[keep] / band["height"], 4)


# ---------------------------------------------------------------- 2. figure 1: parallel coordinates
def q_parallel_coordinates():
    """One line per well, crossing every variable. Plotly lets the reader drag on any axis
    to keep only the wells inside that range, and the filter survives the HTML export."""
    br_levels = Q_LEVELS["Br"]
    br_shades = q_level_shades(len(br_levels))
    br_index = q_wells["Br_design_percent"].map({level: i for i, level in enumerate(br_levels)})
    # A stepped colour scale: each bromide level gets exactly one shade.
    steps = len(br_levels)
    colorscale = []
    for i, shade in enumerate(br_shades):
        colorscale += [[i / steps, shade], [(i + 1) / steps, shade]]

    def knob_axis(knob):
        levels = Q_LEVELS[knob]
        return {
            "label": f"{knob} level (%)", "values": q_wells[f"{knob}_design_percent"],
            "tickvals": levels, "ticktext": [q_level_text(v) for v in levels],
            "range": [levels[0], levels[-1]],
        }

    # The three measured wavelengths share ONE range, so a horizontal segment between two
    # of these axes means "same wavelength" and a slope means "different wavelength".
    measured = q_wells[["abs_edge_nm", "Bottom_centre_nm", "Top_centre_nm"]]
    low = float(np.floor(measured.min().min() / 10) * 10)
    high = float(np.ceil(measured.max().max() / 10) * 10)
    # The two difference axes also share ONE range, so the baseline and the robust
    # answer to "do Top and Bottom differ?" can be compared directly.
    differences = q_wells[["Top_minus_Bottom_peak_nm", "delta_centre_nm"]]
    d_low = float(np.floor(differences.min().min() / 10) * 10)
    d_high = float(np.ceil(differences.max().max() / 10) * 10)

    figure = go.Figure(go.Parcoords(
        line={"color": br_index, "colorscale": colorscale, "cmin": -0.5, "cmax": steps - 0.5,
              "showscale": False},
        dimensions=[knob_axis(knob) for knob in Q_KNOBS] + [
            {"label": "Absorption<br>edge (nm)", "values": q_wells["abs_edge_nm"], "range": [low, high]},
            {"label": "Bottom PL<br>band (nm)", "values": q_wells["Bottom_centre_nm"], "range": [low, high]},
            {"label": "Top PL<br>band (nm)", "values": q_wells["Top_centre_nm"], "range": [low, high]},
            {"label": "Top − Bottom<br>baseline peak (nm)", "values": q_wells["Top_minus_Bottom_peak_nm"],
             "range": [d_low, d_high]},
            {"label": "Top − Bottom<br>area centre (nm)", "values": q_wells["delta_centre_nm"],
             "range": [d_low, d_high]},
        ],
        labelfont={"size": 12, "color": SOLUTION_INK["text"]},
        tickfont={"size": 11, "color": SOLUTION_INK["muted"]},
        rangefont={"size": 1, "color": "rgba(0,0,0,0)"},      # the tick labels already give the range
        unselected={"line": {"color": "#c9c8c2", "opacity": 0.25}},
    ))
    chips = "   ".join(
        f"<span style='color:{shade}'>■</span> Br {q_level_text(level)} %"
        for level, shade in zip(br_levels, br_shades)
    )
    figure.add_annotation(
        x=0, y=1.27, xref="paper", yref="paper", xanchor="left", yanchor="bottom", showarrow=False,
        align="left", font={"size": 12.5, "color": SOLUTION_INK["text"]},
        text=f"Line colour = bromide level:   {chips}",
    )
    # Brackets under the axes say which axes share a scale.
    for start, stop, text in [
        (0, 3, "composition<br>(what was dispensed)"),
        (4, 6, "measured wavelengths<br>(one shared scale)"),
        (7, 8, "Top − Bottom, two ways<br>(one shared scale)"),
    ]:
        figure.add_annotation(
            x=(start + stop) / 16, y=-0.13, xref="paper", yref="paper", showarrow=False,
            yanchor="top", text=text, font={"size": 11.5, "color": SOLUTION_INK["muted"]},
        )
    figure.update_layout(
        title={
            "text": (
                "Every well through every variable: composition → absorption edge → PL band → Top vs Bottom"
                "<br><sup>One line = one well (96 lines). Drag up or down along any axis to keep only the "
                "wells in that range; click the axis again to clear. Flat segments between the wavelength "
                "axes mean the measurements agree.</sup>"
            ),
            "x": 0.01, "xanchor": "left", "y": 0.965, "yanchor": "top",
        },
        template="plotly_white", height=640, autosize=True,
        margin={"l": 70, "r": 70, "t": 190, "b": 100},
        font={"color": SOLUTION_INK["text"]}, plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb",
    )
    return figure


# ---------------------------------------------------------------- 3. figure 2: mirror spectra
def q_mirror_spectra():
    """All 96 full spectra. Top is drawn upward and Bottom downward from a shared zero line,
    so two geometries that agree look like mirror images of each other."""
    br_levels = Q_LEVELS["Br"]
    figure = make_subplots(
        rows=1, cols=len(br_levels), shared_yaxes=True, horizontal_spacing=0.02,
        subplot_titles=[f"Br {q_level_text(level)} % of halide" for level in br_levels],
    )
    owners = []                                   # one (well, geometry) entry per spectrum trace
    for column, br_level in enumerate(br_levels, start=1):
        block = q_wells[q_wells["Br_design_percent"].eq(br_level)]
        for record in block.to_dict("records"):
            for geometry, sign in [("Top", 1.0), ("Bottom", -1.0)]:
                wavelength, value = q_normalised(record["well"], geometry, 600, 850)
                figure.add_trace(go.Scatter(
                    x=wavelength, y=sign * value, mode="lines", showlegend=False,
                    line={"color": SOLUTION_COLORS[geometry], "width": 1}, opacity=0.6,
                    hovertemplate=(
                        f"<b>Well {record['well']} · {geometry} PL</b><br>"
                        f"Cs {q_level_text(record['Cs_design_percent'])} % · "
                        f"Rb {q_level_text(record['Rb_design_percent'])} % · "
                        f"DMA {q_level_text(record['DMA_design_percent'])} %<br>"
                        f"band centre {record[geometry + '_centre_nm']:.1f} nm<br>"
                        "%{x:.0f} nm<extra></extra>"
                    ),
                ), row=1, col=column)
                owners.append((record, geometry))
        # Absorption edges as a row of ticks on the zero line (independent measurement).
        clear = block[~block["edge_weak"]]
        figure.add_trace(go.Scatter(
            x=clear["abs_edge_nm"], y=np.zeros(len(clear)), mode="markers",
            name="Absorption edge of each well", showlegend=column == 1,
            marker={"symbol": "line-ns-open", "size": 13, "color": SOLUTION_COLORS["Absorbance"],
                    "line": {"width": 1.6, "color": SOLUTION_COLORS["Absorbance"]}},
            customdata=clear[["well"]],
            hovertemplate="Well %{customdata[0]}<br>absorption edge %{x:.0f} nm<extra></extra>",
        ), row=1, col=column)
        # Point at the wells where the baseline's single highest point is a noise spike.
        for record in block[block["naive_artifact"]].to_dict("records"):
            axis = f"x{column if column > 1 else ''}"
            # put the note on the empty side of the panel, away from the band
            note_x = 606 if record["Bottom_centre_nm"] > 725 else 757
            figure.add_annotation(
                x=849, y=1.26, xref=axis, yref="y", ax=note_x, ay=1.08, axref=axis, ayref="y",
                text=(f"<b>{record['well']}</b>: edge spike =<br>baseline's Top “peak”<br>"
                      f"({signed_text(record['Top_minus_Bottom_peak_nm'])} nm vs Bottom)"),
                showarrow=True, arrowhead=2, arrowwidth=1, arrowcolor=SOLUTION_INK["muted"],
                xanchor="left", align="left", font={"size": 10.5, "color": SOLUTION_INK["muted"]},
                bgcolor="rgba(252,252,251,0.85)",
            )
    spectrum_count = len(owners)

    # Buttons recolour the same lines, so the A-site knobs can be read from the full spectra too.
    def colours_by(knob):
        if knob is None:
            return [SOLUTION_COLORS[geometry] for _, geometry in owners]
        shades = dict(zip(Q_LEVELS[knob], q_level_shades(len(Q_LEVELS[knob]))))
        return [shades[record[f"{knob}_design_percent"]] for record, _ in owners]

    def key_text(knob):
        if knob is None:
            return (f"Line colour:   <span style='color:{SOLUTION_COLORS['Top']}'>■</span> Top PL (drawn upward)"
                    f"   <span style='color:{SOLUTION_COLORS['Bottom']}'>■</span> Bottom PL (drawn downward)")
        chips = "   ".join(
            f"<span style='color:{shade}'>■</span> {q_level_text(level)} %"
            for level, shade in zip(Q_LEVELS[knob], q_level_shades(len(Q_LEVELS[knob])))
        )
        return f"Line colour = {knob} design level:   {chips}   (Top is still upward, Bottom downward)"

    key_index = len(figure.layout.annotations)
    figure.add_annotation(
        x=0, y=1.12, xref="paper", yref="paper", xanchor="left", yanchor="bottom", showarrow=False,
        align="left", font={"size": 12.5, "color": SOLUTION_INK["text"]}, text=key_text(None),
    )
    trace_ids = [i for i, trace in enumerate(figure.data) if trace.mode == "lines"]
    assert len(trace_ids) == spectrum_count
    buttons = [{
        "label": label, "method": "update",
        "args": [{"line.color": colours_by(knob)}, {f"annotations[{key_index}].text": key_text(knob)},
                 trace_ids],
    } for label, knob in [("Colour by geometry", None), ("by DMA level", "DMA"),
                          ("by Cs level", "Cs"), ("by Rb level", "Rb")]]

    figure.update_xaxes(range=[600, 850], gridcolor=SOLUTION_INK["grid"], zeroline=False,
                        tickvals=[650, 700, 750, 800], title_text="Wavelength (nm)")
    figure.update_yaxes(range=[-1.3, 1.3], gridcolor=SOLUTION_INK["grid"], zeroline=True,
                        zerolinecolor=SOLUTION_INK["invalid"], zerolinewidth=1,
                        tickvals=[-1, -0.5, 0, 0.5, 1], ticktext=["1", "0.5", "0", "0.5", "1"])
    figure.update_yaxes(title_text="← Bottom PL    |    Top PL →<br>(each ÷ its band height)", row=1, col=1)
    figure.update_layout(
        updatemenus=[{
            "type": "buttons", "direction": "right", "buttons": buttons, "active": 0,
            "x": 0.0, "xanchor": "left", "y": 1.21, "yanchor": "bottom",
            "bgcolor": "#fcfcfb", "bordercolor": "#c9c8c2", "font": {"size": 12},
            "pad": {"r": 4, "t": 0, "b": 0}, "showactive": True,
        }],
        title={
            "text": (
                "The evidence: all 96 full spectra, Top mirrored against Bottom"
                "<br><sup>32 wells per panel. If the two geometries agree, the upper and lower halves "
                "are mirror images. The spread inside a panel is the A-site effect. Hover any line "
                "for its well.</sup>"
            ),
            "x": 0.01, "xanchor": "left", "y": 0.975, "yanchor": "top",
        },
        template="plotly_white", height=640, autosize=True, hovermode="closest",
        margin={"l": 95, "r": 30, "t": 190, "b": 95},
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": -0.16, "yanchor": "top",
                "font": {"size": 12}},
        font={"color": SOLUTION_INK["text"]}, plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb",
    )
    return figure


# ---------------------------------------------------------------- 4. figure 3: cumulative check
def q_step_curve(values):
    """x and y of a cumulative curve: y = share of wells (%) whose value is at most x."""
    ordered = np.sort(np.asarray(values, float))
    share = 100.0 * np.arange(1, len(ordered) + 1) / len(ordered)
    return np.concatenate([[ordered[0]], ordered]), np.concatenate([[0.0], share])


def q_cumulative():
    limit = 20.0                                   # x-range of the left panel (nm)
    figure = make_subplots(
        rows=1, cols=2, horizontal_spacing=0.11,
        subplot_titles=(
            "How far apart are Top and Bottom?",
            "How far is the PL band from the absorption edge?",
        ),
    )
    gaps = {
        "Area centre (this solution)": (q_wells["delta_centre_nm"].abs(), SOLUTION_LEVEL_RAMP[3], 3),
        "Single highest point (baseline)": (q_wells["Top_minus_Bottom_peak_nm"].abs(), SOLUTION_INK["invalid"], 2.2),
    }
    for name, (values, colour, width) in gaps.items():
        x, y = q_step_curve(values)
        x[0] = 0.0
        figure.add_trace(go.Scatter(
            x=np.clip(x, 0, limit * 2), y=y, mode="lines", line_shape="hv", name=name,
            line={"color": colour, "width": width},
            hovertemplate=name + "<br>%{y:.0f} % of wells are within %{x:.1f} nm<extra></extra>",
        ), row=1, col=1)
    beyond = q_wells[q_wells["Top_minus_Bottom_peak_nm"].abs() > limit].sort_values("Top_minus_Bottom_peak_nm")
    if not beyond.empty:
        listed = ", ".join(f"{w} ({signed_text(d)} nm)" for w, d in
                           zip(beyond["well"], beyond["Top_minus_Bottom_peak_nm"]))
        figure.add_annotation(
            x=limit * 0.98, y=100.0 * (1 - len(beyond) / len(q_wells)) - 9, xref="x", yref="y",
            xanchor="right", yanchor="top", showarrow=False, align="right",
            font={"size": 11.5, "color": SOLUTION_INK["muted"]},
            text=f"<b>baseline:</b> never reaches 100 % here;<br>{listed}<br>lie off this scale",
        )
    worst = float(q_wells["delta_centre_nm"].abs().max())
    figure.add_annotation(
        x=worst, y=100, xref="x", yref="y", ax=8.5, ay=42, axref="x", ayref="y", showarrow=True,
        arrowhead=2, arrowwidth=1, arrowcolor=SOLUTION_INK["muted"], xanchor="left",
        font={"size": 12, "color": SOLUTION_INK["text"]},
        text=f"<b>this solution:</b> all {len(q_wells)} wells<br>within {worst:.1f} nm",
    )

    clear = q_wells[~q_wells["edge_weak"]]
    br_levels = Q_LEVELS["Br"]
    label_spots = np.linspace(0.85, 0.2, len(br_levels))       # stagger the labels down the curves
    for position, (br_level, shade) in enumerate(zip(br_levels, q_level_shades(len(br_levels)))):
        subset = clear[clear["Br_design_percent"].eq(br_level)]
        x, y = q_step_curve(subset["edge_to_pl_nm"])
        figure.add_trace(go.Scatter(
            x=x, y=y, mode="lines", line_shape="hv", name=f"Br {q_level_text(br_level)} %",
            line={"color": shade, "width": 3},
            hovertemplate=(f"Br {q_level_text(br_level)} %<br>%{{y:.0f}} % of wells: PL band at most "
                           "%{x:.1f} nm red of the edge<extra></extra>"),
        ), row=1, col=2)
        spot = float(label_spots[position])
        on_left = position == 0                    # the leftmost curve has free space on its left
        figure.add_annotation(
            x=float(np.quantile(subset["edge_to_pl_nm"], spot)), y=100 * spot, xref="x2", yref="y2",
            showarrow=False, xanchor="right" if on_left else "left", xshift=-8 if on_left else 8,
            text=f"<b>Br {q_level_text(br_level)} %</b>", font={"size": 12, "color": SOLUTION_INK["text"]},
        )
    figure.add_vline(x=0, line={"color": SOLUTION_INK["invalid"], "width": 1}, row=1, col=2)
    figure.update_xaxes(gridcolor=SOLUTION_INK["grid"], zeroline=False)
    figure.update_yaxes(gridcolor=SOLUTION_INK["grid"], zeroline=False, range=[0, 104],
                        tickvals=[0, 25, 50, 75, 100], ticksuffix=" %")
    figure.update_xaxes(title_text="|Top − Bottom| band position (nm)", range=[0, limit], row=1, col=1)
    figure.update_xaxes(title_text="PL band centre − absorption edge (nm)", row=1, col=2)
    figure.update_yaxes(title_text="Share of wells at or below this value", row=1, col=1)
    figure.update_layout(
        title={"text": (
            "The same answer as a share of the plate"
            "<br><sup>Read a curve as: “this share of the wells lies at or below this value”. "
            f"Right panel uses the {len(clear)} wells with a clear absorption edge.</sup>"
        ), "x": 0.01, "xanchor": "left"},
        template="plotly_white", height=540, autosize=True,
        margin={"l": 85, "r": 30, "t": 120, "b": 110},
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": -0.2, "yanchor": "top",
                "font": {"size": 12}},
        font={"color": SOLUTION_INK["text"]}, plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb",
    )
    return figure


challenge1_figures = [q_parallel_coordinates(), q_mirror_spectra(), q_cumulative()]


# Challenge 1 — what the figures show (every number is computed from the data above)
q_br_means = q_wells.groupby("Br_design_percent")["Bottom_centre_nm"].mean()
q_knob_effect = {
    knob: (q_wells[q_wells[f"{knob}_design_percent"].eq(Q_LEVELS[knob][-1])]["Bottom_centre_nm"].mean()
           - q_wells[q_wells[f"{knob}_design_percent"].eq(Q_LEVELS[knob][0])]["Bottom_centre_nm"].mean())
    for knob in ["DMA", "Cs", "Rb"]
}
q_strong_edges = q_wells[~q_wells["edge_weak"]]
q_flagged = q_wells[q_wells["naive_artifact"]]
display(Markdown(f'''
### Challenge 1 — what did we just do?

**Question.** How does composition move the PL band, do Top and Bottom agree about it, and
does the absorption edge tell the same story?

**What the figures show.**

- **Bromide sets the band.** The average Bottom PL band centre steps from
  **{q_br_means.iloc[0]:.0f} nm** to **{q_br_means.iloc[1]:.0f} nm** to
  **{q_br_means.iloc[2]:.0f} nm** as Br goes {" → ".join(q_level_text(v) for v in Q_LEVELS["Br"])} %.
- **The A-site only fine-tunes it.** Lowest to highest design level, averaged over the plate:
  DMA **{signed_text(q_knob_effect["DMA"], 0)} nm**, Cs **{signed_text(q_knob_effect["Cs"], 0)} nm**,
  Rb **{signed_text(q_knob_effect["Rb"], 0)} nm**.
- **Top and Bottom agree in all 96 wells.** With the area-based band centre, Top − Bottom runs from
  **{signed_text(q_wells["delta_centre_nm"].min(), 1)}** to **{signed_text(q_wells["delta_centre_nm"].max(), 1)} nm**
  (median {signed_text(q_wells["delta_centre_nm"].median(), 1)} nm). The baseline's single-highest-point
  peak gave **{signed_text(q_wells["Top_minus_Bottom_peak_nm"].min(), 0)}** to
  **{signed_text(q_wells["Top_minus_Bottom_peak_nm"].max(), 0)} nm** for the same wells.
- **Absorbance agrees.** The absorption edge follows the PL band across the plate
  (correlation {np.corrcoef(q_strong_edges["abs_edge_nm"], q_strong_edges["Bottom_centre_nm"])[0, 1]:.2f}
  over the {len(q_strong_edges)} wells with a clear edge); the PL band sits a median
  {q_strong_edges["edge_to_pl_nm"].median():.0f} nm to the red of it.

**Insight sentence.** Each bromide step moves the emission band by about
{abs(q_br_means.diff().mean()):.0f} nm, the A-site cations shift it by
{max(abs(v) for v in q_knob_effect.values()):.0f} nm or less, and Top and Bottom PL agree to within
{np.ceil(q_wells["delta_centre_nm"].abs().max()):.0f} nm in every well.

**Possible misinterpretation.** The large Top − Bottom "disagreements" on the baseline map
(wells {", ".join(q_flagged["well"]) or "none"}) are single noisy points at the 850 nm end of a
weak Top spectrum, not different emission from the two sides of the film; and because every
spectrum is scaled to its own band height, the figures say nothing about brightness
(Bottom is about {10 ** (q_wells["Bottom_log10_integrated_pl"] - q_wells["Top_log10_integrated_pl"]).median():.0f}×
brighter than Top in instrument units, which may be the reader optics and not the sample).

These are optical observations. None of them is a structural phase assignment.
'''))


# ============================================================================
# 9. CHALLENGE 2 SOLUTION - RP-ADDITIVE SYSTEM  (new)
# ============================================================================

# QUESTION THIS DESIGN ANSWERS
#   Where is the boundary between short-wavelength and long-wavelength emission in
#   composition space, when does it move, and do Top and Bottom tell the same story?
#
# DESIGN (two stacked figures, plain Plotly so they survive the HTML export)
#   1. Moving-boundary map - composition space itself (host mix across, MACl level up).
#                            One line per read time marks the host mix at which the band
#                            centre crosses RP_BOUNDARY_NM. Pale line = first read, dark
#                            line = last read, so the picture shows where the boundary is
#                            and which way it creeps. Bottom and Top get one panel each.
#   2. Animated spectra    - the complete spectra at the host mix the boundary runs through,
#                            for every MACl level, with a time slider and a play button.
#                            Top is drawn upward and Bottom downward (the mirror layout of
#                            Challenge 1); a pale ghost keeps the first read on screen.

# ---------------------------------------------------------------- settings
RP_BAND_WINDOW = (560.0, 840.0)   # band search window; leaves out the signal below 560 nm
RP_VIEW_RANGE = (580.0, 840.0)    # wavelength span drawn in the animated spectra
RP_MIN_SNR = 10.0                 # a band must stand this far above the noise to count
RP_MIN_WIDTH = 8.0                # central-half width below this is a noise spike, not a band (nm)
RP_BOUNDARY_NM = None             # band-centre value that defines the boundary; None = pick it from the data
RP_EVIDENCE_HOST_PERCENT = None   # host mix shown in figure 2; None = the mix the boundary runs through
RP_GEOMETRIES = {"Bottom": "Bottom PL", "Top": "Top PL"}

# ---------------------------------------------------------------- 1. band features for every spectrum
rp_times = np.sort(rp_features["time_min"].unique())
rp_time_step = float(np.median(np.diff(rp_times)))
rp_band_rows = [
    band_by_area(rp_wavelengths, spectrum, RP_BAND_WINDOW)
    for spectrum in rp_spectra_by_id.loc[rp_features["spectrum_id"], rp_wavelength_columns].to_numpy(float)
]
rp_band = pd.concat(
    [rp_features.reset_index(drop=True), pd.DataFrame(rp_band_rows)], axis=1
)
# Three honest states for every spectrum - "no band" and "invalid" are NOT plotted as zero.
rp_band["state"] = np.where(
    rp_band["qc_status"].ne("valid"), "invalid",
    np.where(
        (rp_band["snr"] < RP_MIN_SNR) | (rp_band["width_nm"] < RP_MIN_WIDTH) | rp_band["at_edge"],
        "no band", "band",
    ),
)

# One row per well: composition, how many usable reads, where the band starts and ends.
rp_levels = {
    "host": sorted(rp_composition["FAPbI3_percent"].unique()),
    "macl": sorted(rp_composition["additive_percent"].unique()),
}


def rp_well_summary(group):
    row = {}
    paired = {}
    for geometry, measure in RP_GEOMETRIES.items():
        part = group[group["measure"].eq(measure)].sort_values("time_min")
        good = part[part["state"].eq("band")]
        row[f"{geometry}_band_reads"] = len(good)
        row[f"{geometry}_invalid_reads"] = int(part["state"].eq("invalid").sum())
        # early / late = median of the first / last three reads that show a band
        row[f"{geometry}_early_nm"] = good["centre_nm"].iloc[:3].median() if len(good) >= 4 else np.nan
        row[f"{geometry}_late_nm"] = good["centre_nm"].iloc[-3:].median() if len(good) >= 4 else np.nan
        paired[geometry] = good.set_index("time_min")["centre_nm"]
    both = pd.concat(paired, axis=1).dropna()
    row["paired_reads"] = len(both)
    row["delta_centre_nm"] = (both["Top"] - both["Bottom"]).median() if len(both) else np.nan
    return pd.Series(row)


rp_wells = (
    rp_band.groupby("well")[["measure", "time_min", "state", "centre_nm"]]
    .apply(rp_well_summary).reset_index()
    .merge(rp_composition, on="well", validate="one_to_one")
)
for geometry in RP_GEOMETRIES:
    rp_wells[f"{geometry}_shift_nm"] = rp_wells[f"{geometry}_late_nm"] - rp_wells[f"{geometry}_early_nm"]


def rp_host_text(value):
    return f"{float(value):.0f}" if abs(value - round(value)) < 0.05 else f"{float(value):.1f}"


def rp_time_shades(times):
    """One blue per read time: pale at the first read, dark at the last."""
    pale, dark = [np.array([int(c[i:i + 2], 16) for i in (1, 3, 5)], float)
                  for c in (SOLUTION_LEVEL_RAMP[0], SOLUTION_LEVEL_RAMP[3])]
    fraction = (np.asarray(times, float) - rp_times[0]) / (rp_times[-1] - rp_times[0])
    return [f"rgb({r:.0f},{g:.0f},{b:.0f})" for r, g, b in pale + fraction[:, None] * (dark - pale)]


# ---------------------------------------------------------------- 2. the boundary
# The plate holds two emission families: a short-wavelength one on the 3AMP-rich side and a
# long-wavelength one on the FAPbI3-rich side. The boundary value is set halfway between them.
rp_host_of = rp_wells.set_index("well")["FAPbI3_percent"]
rp_macl_of = rp_wells.set_index("well")["additive_percent"]
rp_good = rp_band[rp_band["state"].eq("band")].assign(
    host=lambda d: d["well"].map(rp_host_of), macl=lambda d: d["well"].map(rp_macl_of)
)
rp_short_family = float(rp_good.loc[rp_good["host"].between(1, 40), "centre_nm"].median())
rp_long_family = float(rp_good.loc[rp_good["host"].between(60, 99), "centre_nm"].median())
if RP_BOUNDARY_NM is None:
    RP_BOUNDARY_NM = float(5 * round((rp_short_family + rp_long_family) / 2 / 5))


def rp_boundary_position(hosts, centres):
    """Host mix (% FAPbI3) at which the band centre crosses RP_BOUNDARY_NM.

    The boundary is the 3AMP-side end of the block of long-wavelength wells. The walk starts
    on the FAPbI3-rich side, steps over wells there that have no band or a short band (pure
    FAPbI3 often loses its emission), finds the long-wavelength block, and follows it down to
    the first well below the boundary value. The position between those two neighbours is
    linear interpolation. Returns NaN when the well needed for that last step has no band.
    """
    in_block = False
    for index in range(len(hosts) - 1, 0, -1):
        here, below = centres[index], centres[index - 1]
        if not in_block:
            if np.isnan(here) or here < RP_BOUNDARY_NM:
                continue                                   # not inside the long-wavelength block yet
            in_block = True
        if np.isnan(below):
            return np.nan                                  # no band next door: leave a gap, never a guess
        if below < RP_BOUNDARY_NM:
            share = (RP_BOUNDARY_NM - below) / (here - below)
            return float(hosts[index - 1] + share * (hosts[index] - hosts[index - 1]))
    return np.nan


rp_boundary_rows = []
for geometry in RP_GEOMETRIES:
    table = rp_good[rp_good["geometry"].eq(geometry)].pivot_table(
        index=["macl", "time_min"], columns="host", values="centre_nm"
    ).reindex(columns=rp_levels["host"])
    for (macl, time_min), centres in table.iterrows():
        rp_boundary_rows.append({
            "geometry": geometry, "macl": macl, "time_min": time_min,
            "boundary_host": rp_boundary_position(rp_levels["host"], centres.to_numpy(float)),
        })
rp_boundary = pd.DataFrame(rp_boundary_rows)


# ---------------------------------------------------------------- 3. figure 1: moving-boundary map
def rp_boundary_map():
    host_levels, macl_levels = rp_levels["host"], rp_levels["macl"]
    reads = len(rp_times)
    row_of = {level: index for index, level in enumerate(macl_levels)}   # MACl levels are unevenly
    figure = make_subplots(                                             # spaced, so rows are evenly spaced
        rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.035,
        subplot_titles=[f"{measure} — where the band centre crosses {RP_BOUNDARY_NM:.0f} nm"
                        for measure in RP_GEOMETRIES.values()],
    )
    shades = rp_time_shades(rp_times)
    for column, geometry in enumerate(RP_GEOMETRIES, start=1):
        # the 96 wells as quiet reference marks that also say how complete each well's data is
        counts = rp_wells[f"{geometry}_band_reads"]
        groups = [
            (counts.eq(reads), "circle", "Well: band in every read", 5),
            (counts.gt(0) & counts.lt(reads), "circle-open", "Well: band in only some reads", 7),
            (counts.eq(0), "x-thin", "Well: no band in any read", 7),
        ]
        for mask, symbol, name, size in groups:
            part = rp_wells[mask]
            figure.add_trace(go.Scatter(
                x=part["FAPbI3_percent"], y=part["additive_percent"].map(row_of), mode="markers",
                name=name, legendgroup=name, showlegend=column == 1,
                marker={"symbol": symbol, "size": size, "color": "#b9b8b2",
                        "line": {"width": 1.3, "color": "#b9b8b2"}},
                customdata=np.column_stack([part["well"], part[f"{geometry}_band_reads"].astype(int),
                                            part[f"{geometry}_invalid_reads"].astype(int)]),
                hovertemplate=(f"<b>Well %{{customdata[0]}}</b> · {geometry} PL<br>"
                               f"reads with a clear band: %{{customdata[1]}}/{reads}<br>"
                               f"reads flagged low/invalid: %{{customdata[2]}}/{reads}<extra></extra>"),
            ), row=1, col=column)
        # one boundary line per read time, pale to dark
        part = rp_boundary[rp_boundary["geometry"].eq(geometry)]
        for time_min, shade in zip(rp_times, shades):
            line = part[part["time_min"].eq(time_min)].sort_values("macl")
            first_or_last = time_min in (rp_times[0], rp_times[-1])
            figure.add_trace(go.Scatter(
                x=line["boundary_host"].round(2), y=line["macl"].map(row_of), mode="lines+markers",
                showlegend=False, connectgaps=False,
                line={"color": shade, "width": 3.2 if first_or_last else 1.6},
                marker={"size": 7 if first_or_last else 4, "color": shade},
                customdata=np.column_stack([line["macl"], np.full(len(line), time_min)]),
                hovertemplate=(f"{geometry} PL · %{{customdata[1]:.0f}} min<br>MACl %{{customdata[0]:g}} %<br>"
                               "boundary at %{x:.1f} % FAPbI3<extra></extra>"),
            ), row=1, col=column)
        # direct labels for the first and last line
        for time_min, anchor, shift in [(rp_times[0], "left", 10), (rp_times[-1], "right", -10)]:
            ends = part[part["time_min"].eq(time_min)].dropna(subset=["boundary_host"]).sort_values("macl")
            if ends.empty:
                continue
            top = ends.iloc[-1]
            figure.add_annotation(
                x=top["boundary_host"], y=row_of[top["macl"]], xref=f"x{column if column > 1 else ''}",
                yref="y", text=f"<b>{time_min:.0f} min</b>", showarrow=False, xanchor=anchor,
                xshift=shift, yshift=13, font={"size": 12, "color": SOLUTION_LEVEL_RAMP[1 if time_min == rp_times[0] else 3]},
            )
        axis = f"x{column if column > 1 else ''}"
        # say in numbers how far the boundary travelled at each MACl level
        for macl in macl_levels:
            track = part[part["macl"].eq(macl)].dropna(subset=["boundary_host"]).sort_values("time_min")
            if len(track) < 2:
                note = "boundary not traceable"
            else:
                first, last = track["boundary_host"].iloc[0], track["boundary_host"].iloc[-1]
                note = (f"{first:.1f} → {last:.1f} %  ({signed_text(last - first, 1)})"
                        f"{'' if len(track) == reads else f', {len(track)}/{reads} reads'}")
            figure.add_annotation(
                x=103, y=row_of[macl], xref=axis, yref="y", text=note, showarrow=False,
                xanchor="right", yanchor="bottom", yshift=5,
                font={"size": 10.5, "color": SOLUTION_INK["muted"]},
            )
        for x, text, anchor in [(2, f"← band below {RP_BOUNDARY_NM:.0f} nm<br>(short-wavelength side)", "left"),
                                (98, f"band above {RP_BOUNDARY_NM:.0f} nm →<br>(long-wavelength side)", "right")]:
            figure.add_annotation(
                x=x, y=len(macl_levels) - 0.45, xref=axis, yref="y", text=text, showarrow=False,
                xanchor=anchor, yanchor="bottom", align=anchor,
                font={"size": 11.5, "color": SOLUTION_INK["muted"]},
            )
    # legend entries for the time shading
    for name, shade, width in [(f"Boundary at the first read ({rp_times[0]:.0f} min)", shades[0], 3.2),
                               ("Boundary at the reads in between (pale → dark)", shades[len(shades) // 2], 1.6),
                               (f"Boundary at the last read ({rp_times[-1]:.0f} min)", shades[-1], 3.2)]:
        figure.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name=name,
                                    line={"color": shade, "width": width}), row=1, col=1)
    figure.update_xaxes(range=[-4, 104], tickvals=host_levels, ticktext=[rp_host_text(h) for h in host_levels],
                        tickfont={"size": 10}, showgrid=False, zeroline=False,
                        title_text="FAPbI3 share of the host (%)  ·  the rest is the 3AMP spacer")
    figure.update_yaxes(range=[-0.6, len(macl_levels) + 0.35], tickvals=list(range(len(macl_levels))),
                        ticktext=[f"{level:g}" for level in macl_levels], showgrid=True,
                        gridcolor="#eceae5", zeroline=False)
    figure.update_yaxes(title_text="Nominal MACl design level (%)", row=1, col=1)
    figure.update_layout(
        title={
            "text": (
                f"The boundary between short- and long-wavelength emission, and how it moves in "
                f"{rp_times[-1]:.0f} minutes"
                f"<br><sup>Composition space: each grey mark is a well. Each blue line joins, for one read time, "
                f"the host mix at which the band centre crosses {RP_BOUNDARY_NM:.0f} nm.</sup>"
                f"<br><sup>{RP_BOUNDARY_NM:.0f} nm is halfway between the two emission families "
                f"({rp_short_family:.0f} and {rp_long_family:.0f} nm). Small grey numbers: boundary position "
                "at the first → last read, in % FAPbI3.</sup>"
            ),
            "x": 0.01, "xanchor": "left", "y": 0.97, "yanchor": "top",
        },
        template="plotly_white", height=680, autosize=True, hovermode="closest",
        margin={"l": 80, "r": 25, "t": 150, "b": 135},
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": -0.2, "yanchor": "top",
                "font": {"size": 12}},
        font={"color": SOLUTION_INK["text"]}, plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb",
    )
    return figure


# ---------------------------------------------------------------- 4. figure 2: animated spectra
def rp_evidence_host():
    """Host mix for figure 2: the design level closest to where the Bottom boundary sits."""
    if RP_EVIDENCE_HOST_PERCENT is not None:
        target = RP_EVIDENCE_HOST_PERCENT
    else:
        target = rp_boundary.loc[rp_boundary["geometry"].eq("Bottom"), "boundary_host"].max()
    return rp_levels["host"][int(np.argmin(np.abs(np.asarray(rp_levels["host"]) - target)))]


def rp_animated_spectra():
    host = rp_evidence_host()
    macl_levels = rp_levels["macl"]
    keep = (rp_wavelengths >= RP_VIEW_RANGE[0]) & (rp_wavelengths <= RP_VIEW_RANGE[1])
    wavelength = rp_wavelengths[keep]
    columns = 4
    rows = int(np.ceil(len(macl_levels) / columns))
    figure = make_subplots(
        rows=rows, cols=columns, shared_xaxes=True, shared_yaxes=True,
        horizontal_spacing=0.018, vertical_spacing=0.11,
        subplot_titles=[f"MACl {level:g} %" for level in macl_levels],
    )
    signs = {"Top": 1.0, "Bottom": -1.0}
    moving = {}                                    # (panel, geometry) -> per-time curves and flags
    for panel, macl in enumerate(macl_levels):
        row, column = panel // columns + 1, panel % columns + 1
        well = rp_wells.loc[rp_wells["FAPbI3_percent"].eq(host) & rp_wells["additive_percent"].eq(macl), "well"].iloc[0]
        for geometry, measure in RP_GEOMETRIES.items():
            reads = rp_band[rp_band["well"].eq(well) & rp_band["measure"].eq(measure)].sort_values("time_min")
            raw = rp_spectra_by_id.loc[reads["spectrum_id"], rp_wavelength_columns].to_numpy(float)[:, keep]
            raw = np.clip(raw - np.nanpercentile(raw, 5, axis=1, keepdims=True), 0, None)
            valid = reads["qc_status"].eq("valid").to_numpy()
            # one scale per well and geometry: the brightest valid read = 1, so dimming and
            # brightening stay visible while the film changes
            scale = max(raw[valid].max() if valid.any() else raw.max(), 1e-9)
            curves = np.round(signs[geometry] * raw / scale, 3)
            first = int(np.argmax(valid))
            figure.add_trace(go.Scatter(          # ghost: the first valid read stays on screen
                x=wavelength, y=curves[first], mode="lines", hoverinfo="skip", showlegend=False,
                line={"color": SOLUTION_PALE[geometry], "width": 3},
            ), row=row, col=column)
            moving[(panel, geometry)] = (well, curves, valid, reads["centre_nm"].to_numpy(float),
                                         reads["state"].to_numpy())
    # the animated traces are added after all ghosts, so their indices are easy to list
    animated = []
    start = len(rp_times) - 1                      # the figure opens on the last read
    for panel, macl in enumerate(macl_levels):
        row, column = panel // columns + 1, panel % columns + 1
        for geometry in RP_GEOMETRIES:
            well, curves, valid, centres, states = moving[(panel, geometry)]
            figure.add_trace(go.Scatter(
                x=wavelength, y=curves[start] if valid[start] else [None] * len(wavelength), mode="lines",
                showlegend=False, line={"color": SOLUTION_COLORS[geometry], "width": 2.2},
                hovertemplate=f"Well {well} · {geometry} PL<br>%{{x:.0f}} nm<extra></extra>",
            ), row=row, col=column)
            animated.append(len(figure.data) - 1)
            figure.add_trace(go.Scatter(          # words instead of a line when the read is flagged
                x=[np.mean(RP_VIEW_RANGE) - 60], y=[0.55 * signs[geometry]], mode="text",
                text=["" if valid[start] else "read flagged<br>low/invalid"], hoverinfo="skip",
                showlegend=False, textfont={"size": 11, "color": SOLUTION_INK["invalid"]},
            ), row=row, col=column)
            animated.append(len(figure.data) - 1)

    frames = []
    for index, time_min in enumerate(rp_times):
        data = []
        for panel in range(len(macl_levels)):
            for geometry in RP_GEOMETRIES:
                well, curves, valid, centres, states = moving[(panel, geometry)]
                data.append(go.Scatter(y=curves[index] if valid[index] else [None] * len(wavelength)))
                data.append(go.Scatter(text=["" if valid[index] else "read flagged<br>low/invalid"]))
        frames.append(go.Frame(name=f"{time_min:.0f}", data=data, traces=animated))
    figure.frames = frames

    for name, geometry in [("Top PL, drawn upward", "Top"), ("Bottom PL, drawn downward", "Bottom")]:
        figure.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name=name,
                                    line={"color": SOLUTION_COLORS[geometry], "width": 2.2}), row=1, col=1)
    figure.add_trace(go.Scatter(
        x=[None], y=[None], mode="lines", name=f"Pale line = the same well at its first valid read",
        line={"color": "#c9c8c2", "width": 3},
    ), row=1, col=1)

    step_settings = {"frame": {"duration": 0, "redraw": True}, "mode": "immediate",
                     "transition": {"duration": 0}}
    figure.update_xaxes(range=list(RP_VIEW_RANGE), tickvals=[600, 700, 800],
                        gridcolor=SOLUTION_INK["grid"], zeroline=False)
    figure.update_yaxes(range=[-1.12, 1.12], tickvals=[-1, -0.5, 0, 0.5, 1],
                        ticktext=["1", "0.5", "0", "0.5", "1"], gridcolor=SOLUTION_INK["grid"],
                        zeroline=True, zerolinecolor=SOLUTION_INK["invalid"], zerolinewidth=1)
    for column in range(1, columns + 1):
        figure.update_xaxes(title_text="Wavelength (nm)", row=rows, col=column)
    for row in range(1, rows + 1):
        figure.update_yaxes(title_text="← Bottom  |  Top →<br>(÷ brightest read)", row=row, col=1)
    figure.update_annotations(font={"size": 13})
    figure.update_layout(
        sliders=[{
            "active": start, "x": 0.2, "len": 0.8, "y": -0.17, "yanchor": "top",
            "pad": {"t": 8, "b": 0},
            "currentvalue": {"prefix": "Elapsed time: ", "suffix": " min", "font": {"size": 14}},
            "steps": [{"label": f"{time_min:.0f}", "method": "animate", "args": [[f"{time_min:.0f}"], step_settings]}
                      for time_min in rp_times],
        }],
        updatemenus=[{
            "type": "buttons", "direction": "left", "showactive": False,
            "x": 0.0, "xanchor": "left", "y": -0.215, "yanchor": "top",
            "bgcolor": "#fcfcfb", "bordercolor": "#c9c8c2", "font": {"size": 12},
            "buttons": [
                {"label": "▶ Play", "method": "animate",
                 "args": [None, {"frame": {"duration": 450, "redraw": True}, "fromcurrent": True,
                                 "transition": {"duration": 0}}]},
                {"label": "❚❚ Pause", "method": "animate", "args": [[None], step_settings]},
            ],
        }],
        title={
            "text": (
                f"The evidence: full spectra at {rp_host_text(host)} % FAPbI3, the host mix the boundary runs through"
                "<br><sup>One panel per MACl level. It plays once when opened; then drag the slider or press Play "
                "to step through the 20 reads. Each well is scaled to its own brightest read, so both shape "
                "and brightness changes show.</sup>"
            ),
            "x": 0.01, "xanchor": "left", "y": 0.975, "yanchor": "top",
        },
        template="plotly_white", height=820, autosize=True, hovermode="closest",
        margin={"l": 95, "r": 25, "t": 110, "b": 190},
        legend={"orientation": "h", "x": 0, "xanchor": "left", "y": -0.09, "yanchor": "top",
                "font": {"size": 12}},
        font={"color": SOLUTION_INK["text"]}, plot_bgcolor="#fcfcfb", paper_bgcolor="#fcfcfb",
    )
    return figure


challenge2_figures = [rp_boundary_map(), rp_animated_spectra()]


# Challenge 2 — what the figures show (every number is computed from the data above)
rp_host_shown = rp_evidence_host()
rp_rich = rp_wells[rp_wells["FAPbI3_percent"].gt(rp_host_shown) & rp_wells["FAPbI3_percent"].lt(100)]
rp_low = rp_wells[rp_wells["FAPbI3_percent"].between(20, 40)]
rp_pure = rp_wells[rp_wells["FAPbI3_percent"].eq(100)].sort_values("additive_percent")


def rp_boundary_travel(geometry):
    """First position, last position and travel of the boundary at every MACl level."""
    rows = []
    for macl, track in rp_boundary[rp_boundary["geometry"].eq(geometry)].groupby("macl"):
        track = track.dropna(subset=["boundary_host"]).sort_values("time_min")
        if len(track) >= 2:
            first, last = track["boundary_host"].iloc[0], track["boundary_host"].iloc[-1]
            halfway = track[(track["boundary_host"] - first) * np.sign(last - first) >= 0.5 * abs(last - first)]
            rows.append({"macl": macl, "first": first, "last": last, "travel": last - first,
                         "half_time": float(halfway["time_min"].iloc[0])})
    return pd.DataFrame(rows).set_index("macl")


rp_travel = {geometry: rp_boundary_travel(geometry) for geometry in RP_GEOMETRIES}
rp_mover = rp_travel["Bottom"]["travel"].abs().idxmax()              # MACl level with the largest travel
rp_calm = rp_travel["Bottom"].drop(index=rp_travel["Bottom"]["travel"].abs().nlargest(2).index)
rp_top_reads = rp_band[rp_band["geometry"].eq("Top")]
rp_first_last = (
    rp_band[rp_band["well"].isin(rp_pure["well"]) & rp_band["geometry"].eq("Bottom")]
    .sort_values("time_min").groupby("well")["log10_integrated_pl"].agg(["first", "last"])
)
rp_biggest_drop = (rp_first_last["first"] - rp_first_last["last"]).idxmax()
display(Markdown(f"""
### Challenge 2 — what did we just do?

**Question.** Where is the boundary between short- and long-wavelength emission in composition
space, when does it move, and do Top and Bottom tell the same story?

**What the figures show.**

- **Two emission families.** Hosts with 1 to 40 % FAPbI3 emit around
  **{rp_short_family:.0f} nm** and hosts with 60 to 99 % around **{rp_long_family:.0f} nm**. The boundary is
  where the band centre crosses **{RP_BOUNDARY_NM:.0f} nm**, halfway between them.
- **The boundary moves most at MACl {rp_mover:g} %.** There the Bottom boundary goes from
  **{rp_travel["Bottom"].loc[rp_mover, "first"]:.1f} to {rp_travel["Bottom"].loc[rp_mover, "last"]:.1f} % FAPbI3**
  ({signed_text(rp_travel["Bottom"].loc[rp_mover, "travel"], 1)}), and the Top boundary from
  {rp_travel["Top"].loc[rp_mover, "first"]:.1f} to {rp_travel["Top"].loc[rp_mover, "last"]:.1f} %
  ({signed_text(rp_travel["Top"].loc[rp_mover, "travel"], 1)}). A negative number means the long-wavelength
  side has grown toward the 3AMP-rich side.
- **When.** At that MACl level the Bottom boundary has covered half of its travel by
  **{rp_travel["Bottom"].loc[rp_mover, "half_time"]:.0f} minutes**; the lines on the map are spread out
  from pale to dark instead of jumping in one step.
- **At most other MACl levels it stays put.** Leaving out the two levels with the largest travel,
  the Bottom boundary moves by at most **{rp_calm["travel"].abs().max():.1f} % FAPbI3** in 380 minutes.
- **The FAPbI3-rich side is stable.** For hosts between {rp_host_text(rp_rich["FAPbI3_percent"].min())} and
  {rp_host_text(rp_rich["FAPbI3_percent"].max())} % FAPbI3 the Bottom band centre stays between
  {rp_rich["Bottom_late_nm"].min():.0f} and {rp_rich["Bottom_late_nm"].max():.0f} nm and moves by at most
  {rp_rich["Bottom_shift_nm"].abs().max():.0f} nm.
- **Top and Bottom.** The two boundaries lie a median
  {(rp_travel["Top"]["last"] - rp_travel["Bottom"]["last"]).abs().median():.1f} % FAPbI3 apart at the last
  read. Away from the boundary they separate more: for hosts with
  {rp_host_text(rp_low["FAPbI3_percent"].min())} to {rp_host_text(rp_low["FAPbI3_percent"].max())} % FAPbI3 the
  Top band centre sits a median {signed_text(rp_low["delta_centre_nm"].median())} nm from the Bottom one.
  **{int(rp_top_reads["state"].eq("invalid").sum())} of {len(rp_top_reads)}** Top reads are flagged low/invalid
  by the dataset ({int(rp_band[rp_band["geometry"].eq("Bottom")]["state"].eq("invalid").sum())} of the Bottom
  reads); the map marks every well that lacks a band in some reads, and the animation writes
  "read flagged" where a line would be.
- **Pure FAPbI3 loses its band.** At 100 % FAPbI3 the Bottom band is present in a median
  {rp_pure["Bottom_band_reads"].median():.0f} of 20 reads; well {rp_biggest_drop} drops
  {10 ** (rp_first_last.loc[rp_biggest_drop, "first"] - rp_first_last.loc[rp_biggest_drop, "last"]):.0f}× in
  integrated PL between the first and last read.

**Insight sentence.** The boundary between short- and long-wavelength emission sits near
{rp_travel["Bottom"]["last"].median():.0f} % FAPbI3; at MACl {rp_mover:g} % it starts further toward the
FAPbI3-rich side and creeps back by about {abs(rp_travel["Bottom"].loc[rp_mover, "travel"]):.0f} % FAPbI3 over
the run, in Top and Bottom alike.

**Possible misinterpretation.** The boundary is interpolated between wells that are 9.1 % FAPbI3
apart, so it shows where the change happens and which way it goes, not an exact composition; and
it marks a change in the emitted light, which by itself is not evidence of a structural phase
boundary.
"""))


# ============================================================================
# 10. SHOW (in a notebook) AND EXPORT THE INTERACTIVE HTML FILES
# ============================================================================

if CHOSEN_CHALLENGE not in ("1", "2", "both"):
    raise ValueError('CHOSEN_CHALLENGE must be "1", "2" or "both"')

RESULTS = {
    "1": ("Challenge 1 — Quaternary System", challenge1_figures, "challenge1_quaternary"),
    "2": ("Challenge 2 — RP-Additive System", challenge2_figures, "challenge2_rp_additive"),
}
safe_team = "".join(
    character if character.isalnum() or character in "-_" else "_"
    for character in (TEAM_NAME.strip() or "unnamed_team")
)
output_dir = Path(OUTPUT_FOLDER).resolve()
output_dir.mkdir(parents=True, exist_ok=True)

for key, (challenge_title, figures, stem) in RESULTS.items():
    if CHOSEN_CHALLENGE not in (key, "both"):
        continue
    if IN_NOTEBOOK:
        display(Markdown(f"## {TEAM_NAME or 'Unnamed team'} — {challenge_title}"))
        show_figures(figures)
    export_path = export_figure_grid(
        figures, output_dir / f"{safe_team}_{stem}.html",
        f"{TEAM_NAME or 'Unnamed team'} — {challenge_title}",
    )
    print("Wrote", export_path)

print("Done. Open the .html files above in any browser; they work offline.")

Verified data already present; no download needed.
Challenge 1 — Quaternary System: ready
Challenge 2 — RP-Additive System: ready


### Data ready — all integrity and structure checks passed

,challenge,wells,spectra,times,measurements
0,Challenge 1 — Quaternary System,96,288,none,"Top PL, Bottom PL, absorbance"
1,Challenge 2 — RP-Additive System,96,3840,20,"Top PL, Bottom PL"



### Challenge 1 — what did we just do?

**Question.** How does composition move the PL band, do Top and Bottom agree about it, and
does the absorption edge tell the same story?

**What the figures show.**

- **Bromide sets the band.** The average Bottom PL band centre steps from
  **788 nm** to **745 nm** to
  **701 nm** as Br goes 0 → 16.7 → 33.3 %.
- **The A-site only fine-tunes it.** Lowest to highest design level, averaged over the plate:
  DMA **−15 nm**, Cs **−6 nm**,
  Rb **+6 nm**.
- **Top and Bottom agree in all 96 wells.** With the area-based band centre, Top − Bottom runs from
  **−3.4** to **+2.6 nm**
  (median −1.0 nm). The baseline's single-highest-point
  peak gave **−17** to
  **+155 nm** for the same wells.
- **Absorbance agrees.** The absorption edge follows the PL band across the plate
  (correlation 0.99
  over the 90 wells with a clear edge); the PL band sits a median
  7 nm to the red of it.

**Insight sentence.** Each bromide step moves the emission band by about
44 nm, the A-site cations shift it by
15 nm or less, and Top and Bottom PL agree to within
4 nm in every well.

**Possible misinterpretation.** The large Top − Bottom "disagreements" on the baseline map
(wells D4, H12) are single noisy points at the 850 nm end of a
weak Top spectrum, not different emission from the two sides of the film; and because every
spectrum is scaled to its own band height, the figures say nothing about brightness
(Bottom is about 3×
brighter than Top in instrument units, which may be the reader optics and not the sample).

These are optical observations. None of them is a structural phase assignment.



### Challenge 2 — what did we just do?

**Question.** Where is the boundary between short- and long-wavelength emission in composition
space, when does it move, and do Top and Bottom tell the same story?

**What the figures show.**

- **Two emission families.** Hosts with 1 to 40 % FAPbI3 emit around
  **701 nm** and hosts with 60 to 99 % around **786 nm**. The boundary is
  where the band centre crosses **745 nm**, halfway between them.
- **The boundary moves most at MACl 20 %.** There the Bottom boundary goes from
  **58.0 to 53.8 % FAPbI3**
  (−4.2), and the Top boundary from
  56.0 to 51.0 %
  (−5.0). A negative number means the long-wavelength
  side has grown toward the 3AMP-rich side.
- **When.** At that MACl level the Bottom boundary has covered half of its travel by
  **280 minutes**; the lines on the map are spread out
  from pale to dark instead of jumping in one step.
- **At most other MACl levels it stays put.** Leaving out the two levels with the largest travel,
  the Bottom boundary moves by at most **0.5 % FAPbI3** in 380 minutes.
- **The FAPbI3-rich side is stable.** For hosts between 63.6 and
  90.9 % FAPbI3 the Bottom band centre stays between
  773 and 802 nm and moves by at most
  8 nm.
- **Top and Bottom.** The two boundaries lie a median
  1.2 % FAPbI3 apart at the last
  read. Away from the boundary they separate more: for hosts with
  27.3 to 36.4 % FAPbI3 the
  Top band centre sits a median +19 nm from the Bottom one.
  **370 of 1920** Top reads are flagged low/invalid
  by the dataset (0 of the Bottom
  reads); the map marks every well that lacks a band in some reads, and the animation writes
  "read flagged" where a line would be.
- **Pure FAPbI3 loses its band.** At 100 % FAPbI3 the Bottom band is present in a median
  8 of 20 reads; well F12 drops
  29× in
  integrated PL between the first and last read.

**Insight sentence.** The boundary between short- and long-wavelength emission sits near
45 % FAPbI3; at MACl 20 % it starts further toward the
FAPbI3-rich side and creeps back by about 4 % FAPbI3 over
the run, in Top and Bottom alike.

**Possible misinterpretation.** The boundary is interpolated between wells that are 9.1 % FAPbI3
apart, so it shows where the change happens and which way it goes, not an exact composition; and
it marks a change in the emitted light, which by itself is not evidence of a structural phase
boundary.


## Unnamed team — Challenge 1 — Quaternary System

Wrote /content/hd_optical_outputs/unnamed_team_challenge1_quaternary.html


## Unnamed team — Challenge 2 — RP-Additive System

Wrote /content/hd_optical_outputs/unnamed_team_challenge2_rp_additive.html
Done. Open the .html files above in any browser; they work offline.
